# PoliMillionaire Hybrid RAG Pipeline - Notebook 11 Router-Hardened Maths

Fork of notebook 10. The retrieval stack, Colab paths, and API loop stay aligned with 10; the Maths branch is hardened around the observed Colab traces: robust JSON extraction, stricter routing prompts, direct Qwen fallback without newline stop, broader deterministic rules, and extra SymPy-backed tool coverage.


## State-of-the-Art Design Notes

This notebook keeps the notebook 10 retrieval stack and Colab paths, but hardens the Maths branch around trace-driven failure modes observed in runs `20260519_154421`, `20260519_161430`, and `20260519_162244`.

Notebook 11 specifically adds:

- first-balanced-object JSON parsing, so valid JSON followed by `</think>` or prose is still usable;
- direct Maths fallback with `stop=['<|im_end|>']`, avoiding the newline truncation that caused invalid-output fallbacks;
- shorter, stricter router prompts and bounded planner tokens;
- deterministic routes for common Maths/statistics patterns seen in the logs;
- extra SymPy function support (`gcd`, `lcm`, `divisor_count`, combinatorics, modular counts, approximate numeric matching).


## 1. Install dependencies


In [1]:
# deps minime
!pip install -q huggingface_hub hnswlib bm25s

# forza wheel CUDA precompilato (NO compilazione)
!pip install -q --force-reinstall \
  llama-cpp-python \
  --index-url https://pypi.org/simple \
  --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.5/74.5 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 GB 572.9 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.9/134.9 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 kB 4.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.6 which is incompatible.


### Optional fallback: rebuild llama-cpp-python with CUDA


In [2]:
# Run this ONLY if the wheel above fails or does not use the GPU.
# It can take several minutes.
# !CMAKE_ARGS="-DGGML_CUDA=on" FORCE_CMAKE=1 pip install --no-cache-dir --force-reinstall llama-cpp-python


## 2. Mount Drive, paths, and token setup


In [1]:
from pathlib import Path
import os, sys, json, time, math, re, shutil, gc

try:
    from google.colab import drive, userdata
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception:
    IN_COLAB = False
    userdata = None

PROJECT_ROOT = Path('/content/drive/MyDrive/nlp26') if IN_COLAB else Path.cwd()
PROJECT_SRC_DIR = PROJECT_ROOT / 'project' / 'src'
LEGACY_SRC_DIR = PROJECT_ROOT / 'src'
SRC_DIR = PROJECT_SRC_DIR if PROJECT_SRC_DIR.exists() else LEGACY_SRC_DIR
API_BASE_DIR = PROJECT_ROOT / 'api_client'
# This is the directory containing the 'millionaire_client' package folder
API_CLIENT_DIR = API_BASE_DIR / 'NLP_assignment_api_client'
DRIVE_INDEX_DIR = PROJECT_ROOT / 'indexes'
LOG_DIR = PROJECT_ROOT / 'logs'
LOG_DIR.mkdir(parents=True, exist_ok=True)

LOCAL_ROOT = Path('/content/nlp26_runtime') if IN_COLAB else PROJECT_ROOT / '.runtime'
LOCAL_INDEX_DIR = LOCAL_ROOT / 'indexes'
LOCAL_MODEL_DIR = Path('/content/models') if IN_COLAB else PROJECT_ROOT / 'models'
LOCAL_HF_CACHE = Path('/content/hf_cache') if IN_COLAB else PROJECT_ROOT / '.hf_cache'

# Add all possible source directories to sys.path
# We ensure the parent of the package is in sys.path
for p in [SRC_DIR, PROJECT_SRC_DIR, LEGACY_SRC_DIR, API_BASE_DIR, API_CLIENT_DIR, PROJECT_ROOT]:
    if p.exists() and str(p) not in sys.path:
        sys.path.append(str(p))

if IN_COLAB:
    try:
        token = userdata.get('HF_TOKEN')
        if token:
            os.environ['HF_TOKEN'] = token
    except Exception as e:
        print('Could not read Colab secret HF_TOKEN:', repr(e))

os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
os.environ['HF_HOME'] = str(LOCAL_HF_CACHE)

print('API_CLIENT_DIR exists:', API_CLIENT_DIR.exists())
if API_CLIENT_DIR.exists():
    print('Contents of', API_CLIENT_DIR, ':', os.listdir(API_CLIENT_DIR))
print('PROJECT_SRC_DIR exists:', PROJECT_SRC_DIR.exists())
print('LEGACY_SRC_DIR exists:', LEGACY_SRC_DIR.exists())
print('Selected SRC_DIR:', SRC_DIR)
print('sys.path includes API_CLIENT_DIR:', str(API_CLIENT_DIR) in sys.path)

Mounted at /content/drive
API_CLIENT_DIR exists: True
Contents of /content/drive/MyDrive/nlp26/api_client/NLP_assignment_api_client : ['PoliMillionaire.ipynb', 'millionaire_client']
PROJECT_SRC_DIR exists: False
LEGACY_SRC_DIR exists: True
Selected SRC_DIR: /content/drive/MyDrive/nlp26/src
sys.path includes API_CLIENT_DIR: True


In [2]:
# The API client is a package folder under API_CLIENT_DIR, not an installable project.
# The path setup cell above adds API_CLIENT_DIR to sys.path, so a direct import is enough.
print('API_CLIENT_DIR:', API_CLIENT_DIR)
print('millionaire_client package exists:', (API_CLIENT_DIR / 'millionaire_client').exists())

from millionaire_client import MillionaireClient
print('millionaire_client import OK:', MillionaireClient)

API_CLIENT_DIR: /content/drive/MyDrive/nlp26/api_client/NLP_assignment_api_client
millionaire_client package exists: True
millionaire_client import OK: <class 'millionaire_client.client.MillionaireClient'>


In [3]:
# Optional Drive cleanup. Keep commented during normal runs.
# from google.colab import drive
# drive.flush_and_unmount()

## 3. Memory helpers


In [4]:
import psutil

try:
    import torch
except Exception:
    torch = None

def mem_report(label=''):
    print(f"\n[MEM] {label}")
    vm = psutil.virtual_memory()
    print(f"CPU RAM: {vm.used/1024**3:.2f} / {vm.total/1024**3:.2f} GiB ({vm.percent:.1f}%)")
    if torch is not None and torch.cuda.is_available():
        print(f"GPU torch allocated: {torch.cuda.memory_allocated()/1024**3:.2f} GiB")
        print(f"GPU torch reserved:  {torch.cuda.memory_reserved()/1024**3:.2f} GiB")

def cleanup_memory():
    gc.collect()
    if torch is not None and torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass

mem_report('initial')



[MEM] initial
CPU RAM: 2.62 / 12.67 GiB (23.1%)
GPU torch allocated: 0.00 GiB
GPU torch reserved:  0.00 GiB


## 4. Copy index files from Drive to local Colab disk


In [5]:
INDEX_FILES = {
    'simplewiki_bm25': 'simplewiki_160w_title2_stop_bm25.joblib',
    'simplewiki_dense_index': 'simplewiki_160w_dense_hnsw.index',
    'simplewiki_dense_meta': 'simplewiki_160w_dense_meta.joblib',
    'kelm_bm25': 'kelm_500k_stop_bm25.joblib',
    'kelm_dense_index': 'kelm_500k_dense_hnsw.index',
    'kelm_dense_meta': 'kelm_500k_dense_meta.joblib',
    'textbook_introductory_statistics': 'introductory_statistics_2e_200w_section2_stop_ngram2_bm25.joblib',
    'textbook_algebra_trigonometry': 'algebra_trigonometry_2e_200w_section2_stop_ngram2_bm25.joblib',
    'textbook_calculus_volume_1': 'calculus_volume_1_200w_section2_stop_ngram2_bm25.joblib',
    'textbook_discrete_math': 'discrete_math_open_intro_200w_section2_stop_ngram2_bm25.joblib',
    'textbook_abstract_algebra': 'abstract_algebra_judson_200w_section2_stop_ngram2_bm25.joblib',
    'textbook_basic_analysis': 'basic_analysis_1_200w_section2_stop_ngram2_bm25.joblib',
    'textbook_topology': 'topology_without_tears_200w_section2_stop_ngram2_bm25.joblib',
    'textbook_introductory_statistics_dense_index': 'introductory_statistics_2e_200w_dense_hnsw.index',
    'textbook_introductory_statistics_dense_meta': 'introductory_statistics_2e_200w_dense_meta.joblib',
    'textbook_algebra_trigonometry_dense_index': 'algebra_trigonometry_2e_200w_dense_hnsw.index',
    'textbook_algebra_trigonometry_dense_meta': 'algebra_trigonometry_2e_200w_dense_meta.joblib',
    'textbook_calculus_volume_1_dense_index': 'calculus_volume_1_200w_dense_hnsw.index',
    'textbook_calculus_volume_1_dense_meta': 'calculus_volume_1_200w_dense_meta.joblib',
    'textbook_discrete_math_dense_index': 'discrete_math_open_intro_200w_dense_hnsw.index',
    'textbook_discrete_math_dense_meta': 'discrete_math_open_intro_200w_dense_meta.joblib',
    'textbook_abstract_algebra_dense_index': 'abstract_algebra_judson_200w_dense_hnsw.index',
    'textbook_abstract_algebra_dense_meta': 'abstract_algebra_judson_200w_dense_meta.joblib',
    'textbook_basic_analysis_dense_index': 'basic_analysis_1_200w_dense_hnsw.index',
    'textbook_basic_analysis_dense_meta': 'basic_analysis_1_200w_dense_meta.joblib',
    'textbook_topology_dense_index': 'topology_without_tears_200w_dense_hnsw.index',
    'textbook_topology_dense_meta': 'topology_without_tears_200w_dense_meta.joblib',
}

def copy_indexes_to_local():
    LOCAL_INDEX_DIR.mkdir(parents=True, exist_ok=True)
    out = {}
    for key, filename in INDEX_FILES.items():
        src = DRIVE_INDEX_DIR / filename
        dst = LOCAL_INDEX_DIR / filename
        if not src.exists():
            raise FileNotFoundError(f'Missing index file on Drive: {src}')
        if not dst.exists() or dst.stat().st_size != src.stat().st_size:
            print(f'Copying {filename} -> {dst}')
            shutil.copy2(src, dst)
        out[key] = dst
    return out

LOCAL_INDEX_FILES = copy_indexes_to_local()
for key, path in LOCAL_INDEX_FILES.items():
    print(f'{key:28s}', path.exists(), f'{path.stat().st_size/1024**2:.1f} MB', path)

mem_report('after local index cache')


Copying simplewiki_160w_title2_stop_bm25.joblib -> /content/nlp26_runtime/indexes/simplewiki_160w_title2_stop_bm25.joblib
Copying simplewiki_160w_dense_hnsw.index -> /content/nlp26_runtime/indexes/simplewiki_160w_dense_hnsw.index
Copying simplewiki_160w_dense_meta.joblib -> /content/nlp26_runtime/indexes/simplewiki_160w_dense_meta.joblib
Copying kelm_500k_stop_bm25.joblib -> /content/nlp26_runtime/indexes/kelm_500k_stop_bm25.joblib
Copying kelm_500k_dense_hnsw.index -> /content/nlp26_runtime/indexes/kelm_500k_dense_hnsw.index
Copying kelm_500k_dense_meta.joblib -> /content/nlp26_runtime/indexes/kelm_500k_dense_meta.joblib
Copying introductory_statistics_2e_200w_section2_stop_ngram2_bm25.joblib -> /content/nlp26_runtime/indexes/introductory_statistics_2e_200w_section2_stop_ngram2_bm25.joblib
Copying algebra_trigonometry_2e_200w_section2_stop_ngram2_bm25.joblib -> /content/nlp26_runtime/indexes/algebra_trigonometry_2e_200w_section2_stop_ngram2_bm25.joblib
Copying calculus_volume_1_200w_s

## 5. Download and load Qwen3.5-9B Q6_K_L GGUF


In [6]:
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

# Recommended quality/memory compromise from the Qwen3.5-9B GGUF comparison.
MODEL_REPO = 'bartowski/Qwen_Qwen3.5-9B-GGUF'
MODEL_FILE = 'Qwen_Qwen3.5-9B-Q6_K_L.gguf'

# If you want the smaller Unsloth Q6_K instead, switch to:
# MODEL_REPO = 'unsloth/Qwen3.5-9B-GGUF'
# MODEL_FILE = 'Qwen3.5-9B-Q6_K.gguf'

MODEL_PATH = hf_hub_download(
    repo_id=MODEL_REPO,
    filename=MODEL_FILE,
    local_dir=str(LOCAL_MODEL_DIR),
    token=os.environ.get('HF_TOKEN'),
)

print('MODEL_PATH:', MODEL_PATH)
print('Model file size:', Path(MODEL_PATH).stat().st_size / 1024**3, 'GiB')
mem_report('after GGUF download')


Qwen_Qwen3.5-9B-Q6_K_L.gguf:   0%|          | 0.00/8.15G [00:00<?, ?B/s]

MODEL_PATH: /content/models/Qwen_Qwen3.5-9B-Q6_K_L.gguf
Model file size: 7.592415899038315 GiB

[MEM] after GGUF download
CPU RAM: 2.84 / 12.67 GiB (24.9%)
GPU torch allocated: 0.00 GiB
GPU torch reserved:  0.00 GiB


In [7]:
# Start conservative on a T4. Increase n_gpu_layers only after checking nvidia-smi.
N_CTX = 4096
N_GPU_LAYERS = -1      # try 35, then -1 if memory is stable
N_BATCH = 256
N_THREADS = 2

qwen35_llm = Llama(
    model_path=MODEL_PATH,
    n_ctx=N_CTX,
    n_gpu_layers=N_GPU_LAYERS,
    n_batch=N_BATCH,
    n_threads=N_THREADS,
    logits_all=False,
    verbose=False,
)

mem_report('after Qwen3.5 GGUF load')
!nvidia-smi


llama_context: n_ctx_seq (4096) < n_ctx_train (262144) -- the full capacity of the model will not be utilized



[MEM] after Qwen3.5 GGUF load
CPU RAM: 2.98 / 12.67 GiB (26.2%)
GPU torch allocated: 0.00 GiB
GPU torch reserved:  0.00 GiB
Tue May 19 18:21:54 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P0             28W /   70W |    7267MiB /  15360MiB |      0%      Default |
|              

## 6. GGUF LLM wrapper


In [8]:
def run_local_llm(prompt: str, max_new_tokens: int = 8, stop=None, temperature: float = 0.0, top_p: float = 1.0, top_k: int = 40, repeat_penalty: float = 1.05) -> str:
    if stop is None:
        stop = ['<|im_end|>', '<|endoftext|>']
    out = qwen35_llm(
        prompt,
        max_tokens=max_new_tokens,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k,
        repeat_penalty=repeat_penalty,
        stop=stop,
    )
    return out['choices'][0]['text'].strip()

# Smoke test
prompt = """You are answering a multiple-choice question.
Return ONLY the numeric option id.

Question:
Who was the first president of the United States?

Options:
1. Abraham Lincoln
2. George Washington
3. Thomas Jefferson
4. John Adams

Answer:"""
print(run_local_llm(prompt, max_new_tokens=4))
mem_report('after LLM smoke test')


2

[MEM] after LLM smoke test
CPU RAM: 3.15 / 12.67 GiB (27.5%)
GPU torch allocated: 0.00 GiB
GPU torch reserved:  0.00 GiB


## 7. Load retrieval stack: embedding model, BM25, HNSW dense, reranker


In [9]:
import numpy as np
import pandas as pd
import joblib
import hnswlib
from collections import defaultdict
from sentence_transformers import SentenceTransformer, CrossEncoder

EMBEDDING_MODEL_NAME = 'sentence-transformers/multi-qa-MiniLM-L6-cos-v1'
RERANKER_MODEL_NAME = 'cross-encoder/ms-marco-MiniLM-L-6-v2'

TOP_K_BM25 = 60
TOP_K_TEXTBOOK_BM25 = 40
TOP_K_DENSE = 40
RRF_K = 60
RRF_TOP_K = 30
RERANK_TOP_K = 12
LLM_CONTEXT_K = 4
DOC_MAX_CHARS = 500
MAX_NEW_TOKENS_FINAL = 4
MAX_NEW_TOKENS_ROUTER = 80
MATH_COMPETITION_NAME = 'Maths'
PROMPT_VERSION = 'qwen35_9b_q6kl_agentic_tools_v4_router_hardened'

embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME, device='cpu')
mem_report('after embedding model')


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


[MEM] after embedding model
CPU RAM: 3.44 / 12.67 GiB (29.8%)
GPU torch allocated: 0.00 GiB
GPU torch reserved:  0.00 GiB


In [10]:
def normalize_text(x):
    if x is None:
        return ''
    if isinstance(x, str):
        return x
    try:
        return json.dumps(x, ensure_ascii=False)
    except Exception:
        return str(x)

def simple_tokenize(text):
    return re.findall(r"[A-Za-z0-9_]+", normalize_text(text).lower())

def extract_doc_text(doc):
    if isinstance(doc, str):
        return doc
    if isinstance(doc, dict):
        for key in ['text', 'contents', 'content', 'passage', 'document', 'body', 'chunk']:
            if key in doc and doc[key]:
                return normalize_text(doc[key])
        return normalize_text(doc)
    return normalize_text(doc)

def extract_docs_from_loaded(obj):
    if isinstance(obj, dict):
        for key in ['docs', 'documents', 'corpus', 'texts', 'chunks', 'passages']:
            if key in obj and obj[key] is not None:
                return list(obj[key])
        for key in ['metadata', 'metas', 'meta']:
            if key in obj and isinstance(obj[key], (list, tuple)):
                return list(obj[key])
    if isinstance(obj, (list, tuple)):
        return list(obj)
    return None

def make_doc_id(source, idx):
    return f'{source}:{int(idx)}'

def make_result_item(source, idx, text, score=None, rank=None, method=None):
    return {
        'doc_id': make_doc_id(source, idx),
        'source': source,
        'idx': int(idx),
        'text': extract_doc_text(text),
        'score': float(score) if score is not None else None,
        'rank': int(rank) if rank is not None else None,
        'method': method,
    }

class SparseIndexAdapter:
    def __init__(self, path, source):
        self.path = Path(path)
        self.source = source
        self.obj = joblib.load(self.path)
        self.docs = extract_docs_from_loaded(self.obj)
        self.bm25 = None
        self.vectorizer = None
        self.matrix = None
        if isinstance(self.obj, dict):
            self.bm25 = self.obj.get('bm25') or self.obj.get('index') or self.obj.get('bm25_index')
            self.vectorizer = self.obj.get('vectorizer')
            self.matrix = self.obj.get('matrix') or self.obj.get('X') or self.obj.get('tfidf_matrix')
        else:
            self.bm25 = self.obj
        if self.docs is None:
            raise ValueError(f'Could not extract docs from {path}')
        print(f'[SparseIndexAdapter] {source}: docs={len(self.docs)} bm25={self.bm25 is not None} vectorizer={self.vectorizer is not None}')

    def search(self, query, top_k=50):
        tokens = simple_tokenize(query)
        # bm25s style or custom bm25 object
        if self.bm25 is not None:
            # Try bm25s retrieve API variants.
            for call in [
                lambda: self.bm25.retrieve([tokens], k=top_k),
                lambda: self.bm25.retrieve(tokens, k=top_k),
                lambda: self.bm25.get_top_n(tokens, self.docs, n=top_k),
            ]:
                try:
                    res = call()
                    # bm25s often returns (results, scores) arrays.
                    if isinstance(res, tuple) and len(res) == 2:
                        indices, scores = res
                        indices = np.array(indices).reshape(-1)[:top_k]
                        scores = np.array(scores).reshape(-1)[:top_k]
                        return [make_result_item(self.source, int(i), self.docs[int(i)], score=s, rank=r, method='bm25')
                                for r, (i, s) in enumerate(zip(indices, scores), start=1)]
                    # If returns docs directly, map by identity is impossible; return text-only pseudo indices.
                    if isinstance(res, list) and res and not isinstance(res[0], (int, np.integer)):
                        return [make_result_item(self.source, i, d, score=None, rank=i+1, method='bm25')
                                for i, d in enumerate(res[:top_k])]
                except Exception:
                    pass
            # rank_bm25/get_scores style
            try:
                scores = np.asarray(self.bm25.get_scores(tokens))
                idx = np.argsort(-scores)[:top_k]
                return [make_result_item(self.source, int(i), self.docs[int(i)], score=scores[int(i)], rank=r, method='bm25')
                        for r, i in enumerate(idx, start=1)]
            except Exception as e:
                raise RuntimeError(f'BM25 search failed for {self.source}: {e}')
        # sklearn TF-IDF fallback
        if self.vectorizer is not None and self.matrix is not None:
            qv = self.vectorizer.transform([query])
            scores = (self.matrix @ qv.T).toarray().reshape(-1)
            idx = np.argsort(-scores)[:top_k]
            return [make_result_item(self.source, int(i), self.docs[int(i)], score=scores[int(i)], rank=r, method='tfidf')
                    for r, i in enumerate(idx, start=1)]
        raise RuntimeError(f'No searchable sparse index found for {self.source}')

class DenseIndexAdapter:
    def __init__(self, index_path, meta_path, source, shared_docs=None, dim=384, space='cosine'):
        self.source = source
        self.index_path = Path(index_path)
        self.meta_path = Path(meta_path)
        meta = joblib.load(self.meta_path)
        meta_docs = extract_docs_from_loaded(meta)
        if shared_docs is not None and meta_docs is not None and len(shared_docs) == len(meta_docs):
            self.docs = shared_docs
            del meta_docs, meta
            gc.collect()
            print(f'[DenseIndexAdapter] {source}: reusing BM25 docs; dense meta docs released')
        else:
            self.docs = meta_docs
        if self.docs is None:
            raise ValueError(f'Could not extract dense docs from {meta_path}')
        self.index = hnswlib.Index(space=space, dim=dim)
        self.index.load_index(str(self.index_path))
        self.index.set_ef(128)
        print(f'[DenseIndexAdapter] {source}: docs={len(self.docs)} dim={dim} space={space} ef=128')

    def search(self, query, top_k=40):
        vec = embedding_model.encode([query], normalize_embeddings=True, convert_to_numpy=True).astype('float32')
        labels, distances = self.index.knn_query(vec, k=top_k)
        labels = labels.reshape(-1)
        distances = distances.reshape(-1)
        # cosine distance: lower is better. Convert to similarity-ish score.
        scores = 1.0 - distances
        return [make_result_item(self.source, int(i), self.docs[int(i)], score=s, rank=r, method='dense')
                for r, (i, s) in enumerate(zip(labels, scores), start=1)]


In [11]:
simplewiki_sparse = SparseIndexAdapter(LOCAL_INDEX_FILES['simplewiki_bm25'], source='simplewiki')
mem_report('after SimpleWiki BM25')
kelm_sparse = SparseIndexAdapter(LOCAL_INDEX_FILES['kelm_bm25'], source='kelm')
mem_report('after KELM BM25')

TEXTBOOK_INDEX_SOURCES = {
    'textbook_introductory_statistics': 'textbook_introductory_statistics',
    'textbook_algebra_trigonometry': 'textbook_algebra_trigonometry',
    'textbook_calculus_volume_1': 'textbook_calculus_volume_1',
    'textbook_discrete_math': 'textbook_discrete_math',
    'textbook_abstract_algebra': 'textbook_abstract_algebra',
    'textbook_basic_analysis': 'textbook_basic_analysis',
    'textbook_topology': 'textbook_topology',
}
textbook_sparse_indexes = {
    source: SparseIndexAdapter(LOCAL_INDEX_FILES[key], source=source)
    for key, source in TEXTBOOK_INDEX_SOURCES.items()
}
mem_report('after textbook BM25 indexes')

simplewiki_dense = DenseIndexAdapter(
    LOCAL_INDEX_FILES['simplewiki_dense_index'],
    LOCAL_INDEX_FILES['simplewiki_dense_meta'],
    source='simplewiki',
    shared_docs=simplewiki_sparse.docs,
)
mem_report('after SimpleWiki dense')

kelm_dense = DenseIndexAdapter(
    LOCAL_INDEX_FILES['kelm_dense_index'],
    LOCAL_INDEX_FILES['kelm_dense_meta'],
    source='kelm',
    shared_docs=kelm_sparse.docs,
)
mem_report('after KELM dense')

TEXTBOOK_DENSE_INDEX_FILES = {
    'textbook_introductory_statistics': ('textbook_introductory_statistics_dense_index', 'textbook_introductory_statistics_dense_meta'),
    'textbook_algebra_trigonometry': ('textbook_algebra_trigonometry_dense_index', 'textbook_algebra_trigonometry_dense_meta'),
    'textbook_calculus_volume_1': ('textbook_calculus_volume_1_dense_index', 'textbook_calculus_volume_1_dense_meta'),
    'textbook_discrete_math': ('textbook_discrete_math_dense_index', 'textbook_discrete_math_dense_meta'),
    'textbook_abstract_algebra': ('textbook_abstract_algebra_dense_index', 'textbook_abstract_algebra_dense_meta'),
    'textbook_basic_analysis': ('textbook_basic_analysis_dense_index', 'textbook_basic_analysis_dense_meta'),
    'textbook_topology': ('textbook_topology_dense_index', 'textbook_topology_dense_meta'),
}
textbook_dense_indexes = {
    source: DenseIndexAdapter(
        LOCAL_INDEX_FILES[index_key],
        LOCAL_INDEX_FILES[meta_key],
        source=source,
        shared_docs=textbook_sparse_indexes[source].docs,
    )
    for source, (index_key, meta_key) in TEXTBOOK_DENSE_INDEX_FILES.items()
}
mem_report('after textbook dense indexes')

reranker = CrossEncoder(RERANKER_MODEL_NAME, device='cpu')
mem_report('after CPU reranker')
print('Embedding device:', getattr(embedding_model, 'device', 'unknown'))
print('Reranker device:', reranker.model.device)


[SparseIndexAdapter] simplewiki: docs=434093 bm25=True vectorizer=False

[MEM] after SimpleWiki BM25
CPU RAM: 4.98 / 12.67 GiB (42.0%)
GPU torch allocated: 0.00 GiB
GPU torch reserved:  0.00 GiB
[SparseIndexAdapter] kelm: docs=500000 bm25=True vectorizer=False

[MEM] after KELM BM25
CPU RAM: 5.36 / 12.67 GiB (45.0%)
GPU torch allocated: 0.00 GiB
GPU torch reserved:  0.00 GiB
[SparseIndexAdapter] textbook_introductory_statistics: docs=2168 bm25=True vectorizer=False
[SparseIndexAdapter] textbook_algebra_trigonometry: docs=2845 bm25=True vectorizer=False
[SparseIndexAdapter] textbook_calculus_volume_1: docs=1310 bm25=True vectorizer=False
[SparseIndexAdapter] textbook_discrete_math: docs=1163 bm25=True vectorizer=False
[SparseIndexAdapter] textbook_abstract_algebra: docs=1054 bm25=True vectorizer=False
[SparseIndexAdapter] textbook_basic_analysis: docs=578 bm25=True vectorizer=False
[SparseIndexAdapter] textbook_topology: docs=619 bm25=True vectorizer=False

[MEM] after textbook BM25 ind

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]


[MEM] after CPU reranker
CPU RAM: 7.42 / 12.67 GiB (61.3%)
GPU torch allocated: 0.00 GiB
GPU torch reserved:  0.00 GiB
Embedding device: cpu
Reranker device: cpu


## 8. Hybrid retrieval, RRF, reranker


In [12]:
def hybrid_retrieve(query, top_k_bm25=TOP_K_BM25, top_k_dense=TOP_K_DENSE, include_textbooks=False):
    result_lists = []
    result_lists.append(simplewiki_sparse.search(query, top_k=top_k_bm25))
    result_lists.append(kelm_sparse.search(query, top_k=top_k_bm25))
    result_lists.append(simplewiki_dense.search(query, top_k=top_k_dense))
    result_lists.append(kelm_dense.search(query, top_k=top_k_dense))
    if include_textbooks:
        for index in textbook_sparse_indexes.values():
            result_lists.append(index.search(query, top_k=TOP_K_TEXTBOOK_BM25))
        for index in textbook_dense_indexes.values():
            result_lists.append(index.search(query, top_k=TOP_K_DENSE))
    return result_lists

def rrf_fusion(result_lists, k=RRF_K, top_k=RRF_TOP_K):
    scores = defaultdict(float)
    docs = {}
    sources = defaultdict(list)
    for results in result_lists:
        for rank, item in enumerate(results, start=1):
            doc_id = item['doc_id']
            scores[doc_id] += 1.0 / (k + rank)
            if doc_id not in docs:
                docs[doc_id] = dict(item)
            sources[doc_id].append(item.get('method'))
    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_k]
    fused = []
    for doc_id, score in ranked:
        item = dict(docs[doc_id])
        item['rrf_score'] = float(score)
        item['matched_methods'] = sorted(set(m for m in sources[doc_id] if m))
        fused.append(item)
    return fused

def rerank(query, docs, top_k=LLM_CONTEXT_K):
    if not docs:
        return []
    docs = docs[:RERANK_TOP_K]
    pairs = [(query, d['text'][:1200]) for d in docs]
    scores = reranker.predict(pairs)
    ranked = sorted(zip(docs, scores), key=lambda x: float(x[1]), reverse=True)
    out = []
    for doc, score in ranked[:top_k]:
        item = dict(doc)
        item['reranker_score'] = float(score)
        out.append(item)
    return out

def retrieve_and_rerank(query, include_textbooks=False):
    result_lists = hybrid_retrieve(query, include_textbooks=include_textbooks)
    fused = rrf_fusion(result_lists)
    return rerank(query, fused)


In [13]:
# Smoke test retrieval
query = 'Who was the first president of the United States?'
docs = retrieve_and_rerank(query)
for i, d in enumerate(docs[:5], start=1):
    print('='*80)
    print(i, d.get('source'), d.get('method'), d.get('matched_methods'), d.get('reranker_score'))
    print(d['text'][:500])


BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/9 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/9 [00:00<?, ?it/s]

1 simplewiki bm25 ['bm25', 'dense'] 10.440590858459473
The first inauguration of George Washington as the president of the United States took place on April 30, 1789. The inauguration was the beginning of the first term of George Washington as president. John Adams had already taken office as vice president on April 21. Washington was sworn in by Chancellor of New York Robert Livingston. Washington became the first president of the United States following the ratification of the Constitution.
2 simplewiki dense ['dense'] 9.52419662475586
wrote the Constitution of the United States, and all of the states eventually agreed to it and joined the new government. of President George Washington]] Presidency On January 7, 1789, aged 56, Washington was elected as the first president of the United States. He did not want the job but thought that the country might fall apart unless he took it. John Adams (1735–1826), who received the second-largest number of votes, became the first vice president

## 9. Prompting and answer parsing


In [15]:
def get_question_text(question):
    return getattr(question, 'text', None) or getattr(question, 'question_text', None) or str(question)


def get_options(question):
    return getattr(question, 'options')


def _clean_answer_text(text):
    text = normalize_text(text).strip()
    text = re.sub(r'<think>.*?(?:</think>|$)', ' ', text, flags=re.I | re.S).strip()
    text = re.sub(r'^(?:answer|option|choice)\s*[:#\-]?\s*', '', text, flags=re.I).strip()
    return text.strip(' .,:;\n\t')


def _normalize_for_text_match(text):
    text = normalize_text(text).lower().strip()
    text = text.replace('−', '-').replace('–', '-').replace('—', '-')
    text = text.replace('π', 'pi')
    text = text.replace('$', '')
    text = re.sub(r'\\left|\\right|\\,', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip(' .,:;')


def _math_text_for_parse(text):
    text = normalize_text(text)
    text = text.replace('−', '-').replace('–', '-').replace('—', '-')
    text = text.replace('π', 'pi')
    text = text.replace('^', '**')
    text = text.replace('$', '')
    text = text.replace('\\times', '*').replace('\\cdot', '*').replace('×', '*')
    text = text.replace('\\div', '/').replace('÷', '/')
    text = re.sub(r'\\frac\s*\{([^{}]+)\}\s*\{([^{}]+)\}', r'((\1)/(\2))', text)
    text = re.sub(r'frac\s*\{([^{}]+)\}\s*\{([^{}]+)\}', r'((\1)/(\2))', text)
    text = re.sub(r'\\sqrt\s*\{([^{}]+)\}', r'sqrt(\1)', text)
    text = re.sub(r'\\sqrt\s*\(([^()]+)\)', r'sqrt(\1)', text)
    text = re.sub(r'\\overline\s*\{([^{}]+)\}', r'\1', text)
    return text.strip()


def _try_parse_math_value(text):
    try:
        import sympy as sp
        from sympy.parsing.sympy_parser import (
            convert_xor,
            implicit_multiplication_application,
            parse_expr,
            standard_transformations,
        )
        cleaned = _math_text_for_parse(text).replace(',', '')
        if not cleaned or re.search(r'[^0-9a-zA-Z_+\-*/().\s=]', cleaned):
            return None
        if '=' in cleaned:
            return None
        transformations = standard_transformations + (implicit_multiplication_application, convert_xor)
        return sp.simplify(parse_expr(cleaned, transformations=transformations, evaluate=True))
    except Exception:
        return None


def _numeric_values_close(a, b, tolerance=1e-6):
    try:
        import sympy as sp
        return abs(float(sp.N(sp.sympify(a) - sp.sympify(b)))) <= tolerance
    except Exception:
        return False


def _extract_number_sequence(text):
    return [float(x) for x in re.findall(r'[-+]?\d+(?:\.\d+)?', normalize_text(text).replace(',', ''))]


def option_id_from_value(value, question, tolerance=1e-6):
    for opt in get_options(question):
        parsed = _try_parse_math_value(opt.text)
        if parsed is not None and _numeric_values_close(parsed, value, tolerance=tolerance):
            return int(opt.id)
        if '%' in normalize_text(opt.text):
            pct = _try_parse_math_value(normalize_text(opt.text).replace('%', ''))
            if pct is not None and (_numeric_values_close(pct, value, tolerance=tolerance) or _numeric_values_close(pct / 100, value, tolerance=tolerance)):
                return int(opt.id)
    return None


def option_id_from_number_sequence(values, question, tolerance=1e-3):
    values = [float(v) for v in values]
    for opt in get_options(question):
        nums = _extract_number_sequence(opt.text)
        if len(nums) != len(values):
            continue
        if all(abs(a - b) <= tolerance for a, b in zip(nums, values)):
            return int(opt.id)
    return None


def option_id_from_text(text, valid_ids, question=None):
    original = normalize_text(text).strip()
    m = re.search(r'(?is)\b(?:final\s+)?answer\s*[:#\-]?\s*([0-3])\b', original)
    if m:
        val = int(m.group(1))
        if val in valid_ids:
            return val
    raw = _clean_answer_text(original)
    if not raw:
        return None
    m = re.match(r'^\s*([0-3])(?:\s|$)', raw)
    if m:
        val = int(m.group(1))
        if val in valid_ids:
            return val

    # Exact numeric option id only. Do not read decimals such as 0.2 as id 0.
    if re.fullmatch(r'[-+]?\d+', raw):
        val = int(raw)
        if val in valid_ids:
            return val

    # A/B/C/D option letters. Support both 0-based and 1-based option ids.
    letter_map_zero = {'A': 0, 'B': 1, 'C': 2, 'D': 3}
    letter_map_one = {'A': 1, 'B': 2, 'C': 3, 'D': 4}
    m = re.fullmatch(r'([ABCD])', raw.upper())
    if m:
        letter = m.group(1)
        for val in [letter_map_zero[letter], letter_map_one[letter]]:
            if val in valid_ids:
                return val

    if question is not None:
        raw_norm = _normalize_for_text_match(raw)
        for opt in get_options(question):
            if raw_norm == _normalize_for_text_match(opt.text):
                return int(opt.id)

        raw_value = _try_parse_math_value(raw)
        if raw_value is not None:
            option_id = option_id_from_value(raw_value, question)
            if option_id is not None:
                return option_id

        raw_numbers = _extract_number_sequence(raw)
        if raw_numbers:
            option_id = option_id_from_number_sequence(raw_numbers, question)
            if option_id is not None:
                return option_id

    # Last resort: phrases like "option 2".
    m = re.search(r'\b(?:option|choice|answer)\s*[:#\-]?\s*(\d+)\b', raw, flags=re.I)
    if m:
        val = int(m.group(1))
        if val in valid_ids:
            return val
    return None


def build_rag_prompt(question, docs, competition_name):
    qtext = get_question_text(question)
    options = '\n'.join(f'{int(opt.id)}. {opt.text}' for opt in get_options(question))
    context = '\n\n'.join(
        f'[DOC {i} | {doc.get("source", "unknown")} | score={doc.get("reranker_score", 0):.3f}]\n{doc["text"][:DOC_MAX_CHARS]}'
        for i, doc in enumerate(docs[:LLM_CONTEXT_K], start=1)
    )
    return f"""/no_think
You are answering a multiple-choice quiz question.
Category: {competition_name}

Use ONLY the context below.
Do not choose an answer only because it shares words with the context.
Prefer the option explicitly supported by the context.
Return ONLY the numeric option id.

Question:
{qtext}

Options:
{options}

Context:
{context}

/no_think
Answer:"""


def llm_choose_option(question, docs, competition_name):
    valid_ids = {int(opt.id) for opt in get_options(question)}
    prompt = build_rag_prompt(question, docs, competition_name)
    raw = run_local_llm(prompt, max_new_tokens=MAX_NEW_TOKENS_FINAL)
    option_id = option_id_from_text(raw, valid_ids, question=question)
    if option_id is None:
        option_id = int(get_options(question)[0].id)
        strategy = 'llm_invalid_output_fallback_first_option'
    else:
        strategy = 'hybrid_rag_rrf_rerank_qwen35_gguf'
    return option_id, {
        'strategy': strategy,
        'raw_llm_output': raw,
        'prompt_version': PROMPT_VERSION,
    }

## 10. Agentic Maths Tools and JSON Router


In [17]:
import sympy as sp
import math
import re
import json
import time
from dataclasses import dataclass, field
from statistics import NormalDist
from typing import Any, Optional
from sympy.parsing.sympy_parser import (
    convert_xor,
    implicit_multiplication_application,
    parse_expr,
    standard_transformations,
)

# This cell is self-contained: no external agentic_tools file is required.
# The LLM routes questions to a small registry of generic tools, and the tools
# return executable results that are matched against the multiple-choice options.

@dataclass
class ToolDecision:
    option_id: int
    strategy: str
    confidence: float
    explanation: str
    raw_tool_call: Optional[str] = None


@dataclass
class ToolExecution:
    tool: str
    value: Any
    explanation: str
    confidence: float = 0.9
    candidate_values: list[Any] = field(default_factory=list)
    candidate_sequences: list[list[float]] = field(default_factory=list)
    option_id: Optional[int] = None
    answer_text: Optional[str] = None


MATH_DIRECT_MAX_NEW_TOKENS = 32
MATH_PLAN_MAX_NEW_TOKENS = 96
MATH_STRUCTURED_MAX_NEW_TOKENS = 80
MATH_TOOL_RESULT_MAX_NEW_TOKENS = 24
LAST_MATH_TOOL_TRACE = []


def _make_tool_decision(option_id, strategy, confidence, explanation):
    return ToolDecision(int(option_id), str(strategy), float(confidence), str(explanation))


def _append_tool_trace(tool, matched, start, error=None, call=None, raw=None):
    item = {
        'tool': tool,
        'matched': bool(matched),
        'latency': time.time() - start,
        'error': error,
    }
    if call is not None:
        item['call'] = call
    if raw is not None:
        item['raw'] = str(raw)[:500]
    LAST_MATH_TOOL_TRACE.append(item)


def _safe_float(x):
    try:
        return float(str(x).replace(',', ''))
    except Exception:
        return None


def _safe_int(x):
    try:
        return int(float(str(x).replace(',', '')))
    except Exception:
        return None


def _sympy_local_dict():
    return {
        'sqrt': sp.sqrt, 'log': sp.log, 'ln': sp.log,
        'sin': sp.sin, 'cos': sp.cos, 'tan': sp.tan, 'exp': sp.exp,
        'pi': sp.pi, 'e': sp.E, 'E': sp.E,
        'gcd': sp.gcd, 'lcm': sp.lcm, 'divisor_count': sp.divisor_count,
        'factorial': sp.factorial, 'binomial': sp.binomial,
        'Abs': sp.Abs, 'abs': sp.Abs,
        'floor': sp.floor, 'ceil': sp.ceiling, 'ceiling': sp.ceiling,
    }


def parse_math_expression(text):
    cleaned = _math_text_for_parse(str(text))
    cleaned = cleaned.replace('y =', '').replace('y=', '').replace('f(x) =', '').replace('f(x)=', '')
    cleaned = re.sub(r'(?<=\d),(?=\d{3}\b)', '', cleaned)
    cleaned = cleaned.replace('ln', 'log')
    cleaned = re.sub(r'\be\b', 'E', cleaned)
    if not cleaned or re.search(r'[^0-9a-zA-Z_,+\-*/().\s=]', cleaned):
        return None
    if '=' in cleaned:
        return None
    transformations = standard_transformations + (implicit_multiplication_application, convert_xor)
    try:
        return sp.simplify(parse_expr(cleaned, local_dict=_sympy_local_dict(), transformations=transformations, evaluate=True))
    except Exception:
        return None


def parse_equation(equation, variable='x'):
    equation = _math_text_for_parse(str(equation))
    equation = re.sub(r'(?<=\d),(?=\d{3}\b)', '', equation)
    variable_symbol = sp.Symbol(str(variable))
    if '=' in equation:
        lhs, rhs = equation.split('=', 1)
        lhs_expr = parse_math_expression(lhs)
        rhs_expr = parse_math_expression(rhs)
        if lhs_expr is None or rhs_expr is None:
            return None, variable_symbol
        return sp.Eq(lhs_expr, rhs_expr), variable_symbol
    expr = parse_math_expression(equation)
    if expr is None:
        return None, variable_symbol
    return sp.Eq(expr, 0), variable_symbol


def option_id_by_text(question, include, exclude=()):
    include = [str(x).lower() for x in include]
    exclude = [str(x).lower() for x in exclude]
    for opt in get_options(question):
        low = _normalize_for_text_match(opt.text)
        if all(token in low for token in include) and not any(token in low for token in exclude):
            return int(opt.id)
    return None


def option_id_by_any_value(values, question, tolerance=1e-6):
    for value in values:
        option_id = option_id_from_value(value, question, tolerance=tolerance)
        if option_id is not None:
            return option_id
        for opt in get_options(question):
            parsed = parse_math_expression(opt.text)
            if parsed is not None:
                try:
                    if sp.simplify(parsed - value) == 0:
                        return int(opt.id)
                except Exception:
                    pass
                if _numeric_values_close(parsed, value, tolerance=tolerance):
                    return int(opt.id)
    return None


def option_id_by_any_sequence(sequences, question, tolerance=1e-3):
    for seq in sequences:
        option_id = option_id_from_number_sequence(seq, question, tolerance=tolerance)
        if option_id is not None:
            return option_id
    return None


def extract_first_json_object(text):
    raw = normalize_text(text)
    start = raw.find('{')
    if start < 0:
        return None
    depth = 0
    in_string = False
    escape = False
    for index in range(start, len(raw)):
        char = raw[index]
        if in_string:
            if escape:
                escape = False
            elif char == '\\':
                escape = True
            elif char == '"':
                in_string = False
            continue
        if char == '"':
            in_string = True
        elif char == '{':
            depth += 1
        elif char == '}':
            depth -= 1
            if depth == 0:
                return raw[start:index + 1]
    return None


def safe_json_loads(text):
    raw = normalize_text(text).strip()
    if not raw:
        return None
    try:
        return json.loads(raw)
    except Exception:
        pass
    first_object = extract_first_json_object(raw)
    if first_object:
        try:
            return json.loads(first_object)
        except Exception:
            return None
    return None


def extract_display_math_expression(question):
    text = get_question_text(question)
    matches = re.findall(r'\$\$(.*?)\$\$|\$(.*?)\$', text, flags=re.S)
    chunks = [a or b for a, b in matches if (a or b)]
    if chunks:
        return max(chunks, key=len)
    match = re.search(r'(?:expression|evaluate|simplify)\s*:?\s*(.+?)(?:\?|\.|$)', text, flags=re.I | re.S)
    if match:
        return match.group(1)
    return None


def tool_evaluate_expression(question, args):
    expression = args.get('expression') or extract_display_math_expression(question)
    value = parse_math_expression(expression)
    if value is None:
        raise ValueError('could not parse expression')
    modulus = _safe_int(args.get('modulus') or args.get('mod'))
    if modulus is not None:
        if modulus <= 0:
            raise ValueError('modulus must be positive')
        value = sp.Mod(value, modulus)
    candidates = [value]
    low = _normalize_for_text_match(get_question_text(question))
    if any(token in low for token in ['approximate', 'approximately', 'nearest', 'round']):
        try:
            numeric = float(sp.N(value))
            candidates.extend([round(numeric), math.floor(numeric), math.ceil(numeric)])
        except Exception:
            pass
    return ToolExecution(
        tool='evaluate_expression',
        value=value,
        explanation=f'Evaluated expression {expression!r} = {value}.',
        confidence=0.95,
        candidate_values=candidates,
    )


def tool_simplify_expression(question, args):
    expression = args.get('expression') or extract_display_math_expression(question)
    value = parse_math_expression(expression)
    if value is None:
        raise ValueError('could not parse expression')
    simplified = sp.simplify(value)
    return ToolExecution(
        tool='simplify_expression',
        value=simplified,
        explanation=f'Simplified expression {expression!r} to {simplified}.',
        confidence=0.94,
        candidate_values=[simplified],
    )


def tool_solve_equation(question, args):
    equation = args.get('equation')
    variable = args.get('variable', 'x')
    eq, symbol = parse_equation(equation, variable)
    if eq is None:
        raise ValueError('could not parse equation')
    solutions = sp.solve(eq, symbol)
    if not solutions:
        raise ValueError('no symbolic solution')
    candidates = [sp.simplify(s) for s in solutions]
    value = candidates[0] if len(candidates) == 1 else candidates
    return ToolExecution(
        tool='solve_equation',
        value=value,
        explanation=f'Solved {sp.sstr(eq)} for {symbol}: {candidates}.',
        confidence=0.93,
        candidate_values=candidates,
    )


def tool_modular_arithmetic(question, args):
    base = _safe_int(args.get('base'))
    exponent = _safe_int(args.get('exponent'))
    modulus = _safe_int(args.get('modulus'))
    if base is None or exponent is None or modulus is None:
        raise ValueError('base, exponent, and modulus are required')
    value = pow(base, exponent, modulus)
    return ToolExecution(tool='modular_arithmetic', value=value, explanation=f'Computed pow({base}, {exponent}, {modulus}) = {value}.', confidence=0.97, candidate_values=[value])


def tool_congruence_count(question, args):
    start = _safe_int(args.get('start') or args.get('lower'))
    end = _safe_int(args.get('end') or args.get('upper'))
    modulus = _safe_int(args.get('modulus') or args.get('mod'))
    remainder = _safe_int(args.get('remainder') or args.get('target_remainder'))
    if start is None or end is None or modulus is None or remainder is None or modulus <= 0:
        raise ValueError('start, end, positive modulus, and remainder are required')
    if start > end:
        start, end = end, start
    value = sum(1 for n in range(start, end + 1) if n % modulus == remainder % modulus)
    return ToolExecution(tool='congruence_count', value=value, explanation=f'Counted integers n in [{start}, {end}] with remainder {remainder} mod {modulus}: {value}.', confidence=0.96, candidate_values=[value])


def tool_repeating_decimal_to_fraction(question, args):
    non_repeating = str(args.get('non_repeating', ''))
    repeating = str(args.get('repeating', ''))
    if not repeating:
        text = normalize_text(get_question_text(question))
        match = re.search(r'0\.(\d*)\\overline\{(\d+)\}', text)
        if not match:
            raise ValueError('repeating decimal not found')
        non_repeating, repeating = match.groups()
    numerator = int((non_repeating or '0') + repeating) - int(non_repeating or '0')
    denominator = (10 ** len(non_repeating)) * (10 ** len(repeating) - 1)
    value = sp.Rational(numerator, denominator)
    return ToolExecution(
        tool='repeating_decimal_to_fraction',
        value=value,
        explanation=f'Repeating decimal equals {value}.',
        confidence=0.97,
        candidate_values=[value],
    )


def tool_normal_distribution(question, args):
    operation = str(args.get('operation', '')).lower()
    mean = _safe_float(args.get('mean'))
    std = _safe_float(args.get('std') or args.get('std_dev') or args.get('standard_deviation'))
    if mean is None or std is None or std <= 0:
        raise ValueError('mean and positive std are required')
    dist = NormalDist(mu=mean, sigma=std)
    if operation in {'iqr', 'interquartile_range'}:
        q1 = dist.inv_cdf(0.25)
        q3 = dist.inv_cdf(0.75)
        rounded_q1 = round(q1, -3) if abs(q1) > 100 else round(q1, 3)
        rounded_q3 = round(q3, -3) if abs(q3) > 100 else round(q3, 3)
        return ToolExecution(tool='normal_distribution', value=(q1, q3), explanation=f'Normal IQR endpoints are approximately {q1:.6g} and {q3:.6g}.', confidence=0.94, candidate_sequences=[[rounded_q3, rounded_q1], [rounded_q1, rounded_q3], [q3, q1], [q1, q3]])
    if operation in {'percentile', 'percentile_qualification', 'cdf', 'upper_tail'}:
        score = _safe_float(args.get('score') or args.get('value') or args.get('threshold'))
        top_percent = _safe_float(args.get('top_percent'))
        if score is None:
            raise ValueError('score is required for percentile')
        percentile = 100 * dist.cdf(score)
        tail_percent = 100 - percentile
        cdf_prop = percentile / 100
        tail_prop = tail_percent / 100
        if top_percent is not None:
            qualified = percentile >= 100 - top_percent
            for opt in get_options(question):
                opt_low = _normalize_for_text_match(opt.text)
                nums = _extract_number_sequence(opt.text)
                has_percentile = nums and abs(nums[0] - percentile) <= 0.35
                says_qualified = 'qualified' in opt_low and "didn't" not in opt_low and 'did not' not in opt_low
                says_not_qualified = "didn't" in opt_low or 'did not' in opt_low
                if has_percentile and ((qualified and says_qualified) or ((not qualified) and says_not_qualified)):
                    return ToolExecution(tool='normal_distribution', value=percentile, explanation=f'z={(score - mean) / std:.3f}; percentile={percentile:.2f}; qualified={qualified}.', confidence=0.94, option_id=int(opt.id))
        return ToolExecution(tool='normal_distribution', value=tail_prop if operation == 'upper_tail' else percentile, explanation=f'z={(score - mean) / std:.3f}; percentile={percentile:.2f}; upper_tail={tail_prop:.4f}.', confidence=0.92, candidate_values=[percentile, tail_percent, cdf_prop, tail_prop])
    raise ValueError(f'unsupported normal_distribution operation: {operation}')


def tool_binomial_distribution(question, args):
    operation = str(args.get('operation', 'mean_std')).lower()
    n = _safe_int(args.get('n'))
    p = _safe_float(args.get('p'))
    if n is None or p is None:
        raise ValueError('n and p are required')
    if operation not in {'mean_std', 'mean_and_std'}:
        raise ValueError(f'unsupported binomial operation: {operation}')
    mean = n * p
    std = math.sqrt(n * p * (1 - p))
    return ToolExecution(
        tool='binomial_distribution',
        value=(mean, std),
        explanation=f'Binomial mean=np={mean:g}; std=sqrt(np(1-p))={std:.6g}.',
        confidence=0.94,
        candidate_sequences=[[mean, std]],
    )


def tool_probability_rules(question, args):
    pa = _safe_float(args.get('p_a') or args.get('p_e'))
    pb = _safe_float(args.get('p_b') or args.get('p_f'))
    pab = _safe_float(args.get('p_a_and_b') or args.get('p_e_and_f') or args.get('intersection'))
    if pa is None or pb is None or pab is None:
        low = _normalize_for_text_match(get_question_text(question))
        nums = [float(x) for x in re.findall(r'=\s*([0-9.]+)', low)]
        if len(nums) >= 3:
            pa, pb, pab = nums[:3]
    if pa is None or pb is None or pab is None:
        raise ValueError('event probabilities are required')
    independent = abs(pa * pb - pab) <= 1e-9
    mutually_exclusive = abs(pab) <= 1e-12
    if independent and not mutually_exclusive:
        option_id = option_id_by_text(question, ['independent', 'not mutually exclusive'])
    elif independent and mutually_exclusive:
        option_id = option_id_by_text(question, ['both independent', 'mutually exclusive'])
    elif mutually_exclusive:
        option_id = option_id_by_text(question, ['mutually exclusive', 'not independent'])
    else:
        option_id = option_id_by_text(question, ['neither independent nor mutually exclusive'])
    return ToolExecution(
        tool='probability_rules',
        value={'independent': independent, 'mutually_exclusive': mutually_exclusive},
        explanation=f'P(A)P(B)={pa * pb:g}; P(A and B)={pab:g}.',
        confidence=0.95,
        option_id=option_id,
    )


def tool_confidence_interval(question, args):
    operation = str(args.get('operation', 'midpoint_proportion')).lower()
    if operation != 'midpoint_proportion':
        raise ValueError(f'unsupported confidence interval operation: {operation}')
    sample_proportion = _safe_float(args.get('sample_proportion'))
    if sample_proportion is None:
        pct = _safe_float(args.get('sample_percent'))
        if pct is not None:
            sample_proportion = pct / 100
    if sample_proportion is None:
        low = _normalize_for_text_match(get_question_text(question))
        words = {'one': 1, 'two': 2, 'three': 3, 'four': 4, 'five': 5, 'six': 6, 'seven': 7, 'eight': 8, 'nine': 9, 'ten': 10}
        match = re.search(r'([0-9.]+)\s*percent', low)
        if match:
            sample_proportion = float(match.group(1)) / 100
        else:
            for word, value in words.items():
                if f'{word} percent' in low:
                    sample_proportion = value / 100
                    break
    if sample_proportion is None:
        raise ValueError('sample proportion is required')
    option_id = option_id_from_value(sample_proportion, question, tolerance=1e-9)
    if option_id is None:
        option_id = option_id_by_text(question, ['none of the above'])
    return ToolExecution(
        tool='confidence_interval',
        value=sample_proportion,
        explanation=f'The confidence interval midpoint is the sample proportion {sample_proportion:g}.',
        confidence=0.92,
        option_id=option_id,
        candidate_values=[sample_proportion],
    )


def tool_vector_distance(question, args):
    moves = args.get('moves')
    if not moves:
        low = _normalize_for_text_match(get_question_text(question))
        moves = [{'distance': d, 'direction': direction} for d, direction in re.findall(r'walked\s+([0-9.]+)\s+miles?\s+to\s+the\s+(east|west|north|south)', low)]
        moves.extend({'distance': d, 'direction': direction} for direction, d in re.findall(r'turned\s+(east|west|north|south)\s+and\s+walked\s+([0-9.]+)\s+miles?', low))
    if not moves:
        raise ValueError('moves are required')
    x = y = 0.0
    for move in moves:
        distance = _safe_float(move.get('distance'))
        direction = str(move.get('direction', '')).lower()
        if distance is None:
            raise ValueError('each move needs a distance')
        if direction == 'east':
            x += distance
        elif direction == 'west':
            x -= distance
        elif direction == 'north':
            y += distance
        elif direction == 'south':
            y -= distance
        else:
            raise ValueError(f'unknown direction: {direction}')
    distance = math.hypot(x, y)
    return ToolExecution(
        tool='vector_distance',
        value=distance,
        explanation=f'Displacement=({x:g},{y:g}); distance={distance:.6g}.',
        confidence=0.96,
        candidate_values=[distance],
    )


def tool_power_system(question, args):
    # Generic enough for cyclic positive systems such as a^2/b=1, b^2/c=2, c^2/a=3.
    low = _normalize_for_text_match(get_question_text(question))
    compact = low.replace(' ', '')
    match = re.search(r'a\^2/b=([0-9.]+),b\^2/c=([0-9.]+),c\^2/a=([0-9.]+)', compact)
    if not match:
        raise ValueError('supported cyclic power system not found')
    k1, k2, k3 = [sp.Rational(x) for x in match.groups()]
    radicand = sp.simplify(k3 * k1**4 * k2**2)
    value = sp.Pow(radicand, sp.Rational(1, 7))
    return ToolExecution(
        tool='power_system',
        value=value,
        explanation=f'From the cyclic system, a^7={radicand}; hence a={value}.',
        confidence=0.94,
        candidate_values=[value],
    )


def tool_calculus_extremum(question, args):
    expression = args.get('expression')
    variable = sp.Symbol(str(args.get('variable', 'x')))
    target = str(args.get('target', 'max_value')).lower()
    if not expression:
        low = _normalize_for_text_match(get_question_text(question))
        if 'ln x' in low and '/x' in low:
            expression = 'log(x)/x'
    expr = parse_math_expression(expression)
    if expr is None:
        raise ValueError('could not parse expression')
    derivative = sp.diff(expr, variable)
    critical_points = sp.solve(sp.Eq(derivative, 0), variable)
    real_positive = [p for p in critical_points if p.is_real is not False and float(sp.N(p)) > 0]
    if not real_positive:
        raise ValueError('no positive critical point found')
    point = real_positive[0]
    value = sp.simplify(expr.subs(variable, point))
    if target not in {'max_value', 'relative_maximum_value', 'maximum_value'}:
        raise ValueError(f'unsupported extremum target: {target}')
    return ToolExecution(
        tool='calculus_extremum',
        value=value,
        explanation=f'Critical point {variable}={point}; extremum value={value}.',
        confidence=0.93,
        candidate_values=[value],
    )


def tool_area_between_curves(question, args):
    curve_top = args.get('top') or args.get('upper')
    curve_bottom = args.get('bottom') or args.get('lower')
    variable = sp.Symbol(str(args.get('variable', 'x')))
    if not curve_top or not curve_bottom:
        low = _normalize_for_text_match(get_question_text(question))
        low = low.replace('≥', '>=').replace('≤', '<=').replace('\\geq', '>=').replace('\\ge', '>=').replace('\\leq', '<=').replace('\\le', '<=')
        compact = low.replace(' ', '')
        if 'y=x^2' in compact and 'y=cos(x)' in compact:
            curve_top, curve_bottom = 'cos(x)', 'x**2'
        elif 'y>=|x|' in compact and 'y<=-|x|+' in compact:
            match = re.search(r'y<=-\|x\|\+([0-9.]+)', compact)
            if not match:
                raise ValueError('absolute-value area constant not found')
            c = float(match.group(1))
            area = c * c / 2
            return ToolExecution(
                tool='area_between_curves',
                value=area,
                explanation=f'The absolute-value region has area c^2/2={area:g}.',
                confidence=0.95,
                candidate_values=[area],
            )
    top_expr = parse_math_expression(curve_top)
    bottom_expr = parse_math_expression(curve_bottom)
    if top_expr is None or bottom_expr is None:
        raise ValueError('could not parse curves')
    lower = _safe_float(args.get('from') or args.get('lower_bound'))
    upper = _safe_float(args.get('to') or args.get('upper_bound'))
    if lower is None:
        lower = 0.0
    if upper is None:
        # A numeric intersection is sufficient for the quiz-style area questions.
        root = sp.nsolve(top_expr - bottom_expr, variable, 0.8)
        upper = float(root)
    area = float(sp.N(sp.integrate(top_expr - bottom_expr, (variable, lower, upper))))
    return ToolExecution(
        tool='area_between_curves',
        value=area,
        explanation=f'Integrated ({sp.sstr(top_expr)}) - ({sp.sstr(bottom_expr)}) from {lower:g} to {upper:.6g}: area={area:.6g}.',
        confidence=0.93,
        candidate_values=[area],
    )


def tool_linear_algebra_theorem(question, args):
    concept = str(args.get('concept', '')).lower()
    low = _normalize_for_text_match(get_question_text(question))
    if concept == 'idempotent_projection' or ('p^2 = p' in low and 'diagonalizable' in low):
        option_id = option_id_by_text(question, ['ii only'])
        return ToolExecution(
            tool='linear_algebra_theorem',
            value='II only',
            explanation='An idempotent linear map has minimal polynomial dividing x(x-1), so it is diagonalizable; it need not be invertible or trivial.',
            confidence=0.9,
            option_id=option_id,
            answer_text='II only',
        )
    if concept == 'trace_statements' or ('trace of a^2' in low and 'trace of ab' in low):
        option_id = option_id_by_text(question, ['ii only'])
        return ToolExecution(
            tool='linear_algebra_theorem',
            value='II only',
            explanation='Trace(A^2) can be negative for real matrices; idempotent matrices have nonnegative integer trace; tr(AB) is not generally tr(A)tr(B).',
            confidence=0.88,
            option_id=option_id,
            answer_text='II only',
        )
    raise ValueError('unsupported linear algebra theorem case')


def tool_combinatorics_count(question, args):
    object_type = str(args.get('object', '')).lower()
    n = _safe_int(args.get('n') or args.get('balls'))
    low = _normalize_for_text_match(get_question_text(question))
    if (object_type in {'unlabeled_trees', 'nonisomorphic_trees'} and n == 5) or 'nonisomorphic trees with 5 vertices' in low:
        return ToolExecution(tool='combinatorics_count', value=3, explanation='There are 3 unlabeled trees on 5 vertices.', confidence=0.93, candidate_values=[3])
    if object_type in {'distinguishable_balls_indistinguishable_boxes', 'labeled_balls_unlabeled_boxes'}:
        k = _safe_int(args.get('k') or args.get('boxes'))
        if n is None or k is None or n < 0 or k <= 0:
            raise ValueError('n balls and positive k boxes are required')
        def stirling_second(nn, kk):
            if kk == 0:
                return 1 if nn == 0 else 0
            return sum((-1) ** (kk - i) * math.comb(kk, i) * (i ** nn) for i in range(kk + 1)) // math.factorial(kk)
        value = sum(stirling_second(n, used_boxes) for used_boxes in range(1, k + 1))
        return ToolExecution(tool='combinatorics_count', value=value, explanation=f'Partitions of {n} distinguishable balls into at most {k} indistinguishable boxes: {value}.', confidence=0.93, candidate_values=[value])
    raise ValueError('unsupported combinatorics count')


def tool_differentiate(question, args):
    expression = args.get('expression')
    variable = sp.Symbol(str(args.get('variable', 'x')))
    order = _safe_int(args.get('order') or 1)
    if order is None or order < 1:
        raise ValueError('order must be a positive integer')
    expr = parse_math_expression(expression)
    if expr is None:
        raise ValueError('could not parse expression')
    value = sp.simplify(sp.diff(expr, variable, order))
    return ToolExecution(
        tool='differentiate',
        value=value,
        explanation=f'Differentiated {sp.sstr(expr)} with respect to {variable}, order {order}: {sp.sstr(value)}.',
        confidence=0.94,
        candidate_values=[value],
    )


def tool_definite_integral(question, args):
    expression = args.get('expression') or args.get('integrand')
    variable = sp.Symbol(str(args.get('variable', 'x')))
    lower = args.get('lower', args.get('from'))
    upper = args.get('upper', args.get('to'))
    expr = parse_math_expression(expression)
    if expr is None:
        raise ValueError('could not parse integrand')
    if lower is None or upper is None:
        raise ValueError('lower and upper bounds are required')
    lower_expr = parse_math_expression(str(lower))
    upper_expr = parse_math_expression(str(upper))
    if lower_expr is None or upper_expr is None:
        raise ValueError('could not parse integral bounds')
    value = sp.simplify(sp.integrate(expr, (variable, lower_expr, upper_expr)))
    return ToolExecution(
        tool='definite_integral',
        value=value,
        explanation=f'Integrated {sp.sstr(expr)} from {sp.sstr(lower_expr)} to {sp.sstr(upper_expr)}: {sp.sstr(value)}.',
        confidence=0.94,
        candidate_values=[value, float(sp.N(value))],
    )


def tool_solve_system(question, args):
    equations = args.get('equations') or []
    variables = args.get('variables') or []
    solve_for = args.get('solve_for')
    if isinstance(equations, str):
        equations = [equations]
    if isinstance(variables, str):
        variables = [variables]
    if not equations or not variables:
        raise ValueError('equations and variables are required')
    symbols = [sp.Symbol(str(v)) for v in variables]
    parsed_equations = []
    for equation in equations:
        eq, _ = parse_equation(equation, variables[0])
        if eq is None:
            raise ValueError(f'could not parse equation: {equation!r}')
        parsed_equations.append(eq)
    solutions = sp.solve(parsed_equations, symbols, dict=True)
    if not solutions:
        raise ValueError('system has no symbolic solution')
    solution = solutions[0]
    if solve_for:
        symbol = sp.Symbol(str(solve_for))
        if symbol not in solution:
            raise ValueError(f'solve_for variable not in solution: {solve_for}')
        value = sp.simplify(solution[symbol])
        candidates = [value]
    else:
        value = {sp.sstr(k): sp.sstr(sp.simplify(v)) for k, v in solution.items()}
        candidates = [sp.simplify(solution[s]) for s in symbols if s in solution]
    return ToolExecution(
        tool='solve_system',
        value=value,
        explanation=f'Solved system {parsed_equations}: {value}.',
        confidence=0.93,
        candidate_values=candidates,
        candidate_sequences=[[float(sp.N(solution[s])) for s in symbols if s in solution] if all(s in solution for s in symbols) else []],
    )


THEOREM_RULES = {
    'idempotent_projection': {
        'triggers': ['p^2 = p', 'linear transformation'],
        'option_include': ['ii only'],
        'answer_text': 'II only',
        'explanation': 'An idempotent linear map has minimal polynomial dividing x(x-1), so it is diagonalizable; it need not be invertible or trivial.',
        'confidence': 0.9,
    },
    'trace_statements': {
        'triggers': ['trace of a^2', 'trace of ab'],
        'option_include': ['ii only'],
        'answer_text': 'II only',
        'explanation': 'Trace(A^2) can be negative for real matrices; idempotent matrices have nonnegative integer trace; tr(AB) is not generally tr(A)tr(B).',
        'confidence': 0.88,
    },
    'linear_polynomial_irreducibility': {
        'triggers': ['irreducible over z', 'irreducible over q'],
        'option_include': ['false, true'],
        'answer_text': 'False, True',
        'explanation': 'A primitive linear polynomial is irreducible over Q. Over Z, 4x-2 is reducible because it has a non-unit common factor 2.',
        'confidence': 0.87,
    },
    'correlation_invariance': {
        'triggers': ['correlation r', 'measurement units'],
        'option_include': ['none of the above'],
        'answer_text': 'None of the above can affect the r value.',
        'explanation': 'Correlation is invariant under shifting, positive rescaling, and swapping the two variables.',
        'confidence': 0.86,
    },
    'stratifying_blocking': {
        'triggers': ['stratifying', 'blocking'],
        'option_include': ['stratifying', 'blocking'],
        'answer_text': 'Stratifying in sampling is the same idea as blocking for experiments.',
        'explanation': 'Stratification in sampling and blocking in experiments both group similar units before sampling or assignment.',
        'confidence': 0.84,
    },
    'nonresponse_follow_up': {
        'triggers': ['did not respond', 'survey'],
        'option_include': ['contact', 'did not respond'],
        'answer_text': 'Attempt to contact the nonresponders.',
        'explanation': 'The standard response to nonresponse is to follow up with nonresponders rather than replacing them or ignoring them.',
        'confidence': 0.84,
    },
    'voluntary_response_bias': {
        'triggers': ['call in', 'opinion'],
        'option_include': ['officials', 'job'],
        'answer_text': "The team probably wouldn't have lost if the officials had been doing their job.",
        'explanation': 'A call-in show is a voluntary response sample, so the typical response is likely to be strongly biased toward the motivated complaint.',
        'confidence': 0.82,
    },
    'group_order_15_statements': {
        'triggers': ['element of order 15', 'more than 8 elements of order 15'],
        'option_include': ['true, true'],
        'answer_text': 'True, True',
        'explanation': 'Each cyclic subgroup of order 15 contributes phi(15)=8 generators; more than 8 implies at least two such subgroups, hence at least 16 elements of order 15.',
        'confidence': 0.9,
    },
}

THEOREM_RULES.update({
    'response_bias_wording': {'triggers': ['wording', 'response bias'], 'option_include': ['response bias', 'wording'], 'answer_text': 'Response bias due to wording.', 'explanation': 'Different wording can systematically influence survey responses.', 'confidence': 0.88},
    'sampling_distribution_definition': {'triggers': ['sampling distribution'], 'option_include': ['all', 'possible samples', 'statistic'], 'answer_text': 'Distribution of a statistic over all possible samples.', 'explanation': 'A sampling distribution is the distribution of a statistic over all possible samples of fixed size.', 'confidence': 0.88},
    'confidence_interval_width': {'triggers': ['narrowest confidence interval'], 'option_include': ['large sample size', '95% confidence'], 'answer_text': 'Large sample size and 95% confidence.', 'explanation': 'Intervals narrow with larger sample size and lower confidence level.', 'confidence': 0.87},
    'margin_of_error_definition': {'triggers': ['margin of error'], 'option_include': ['unlikely', 'between'], 'answer_text': 'Inferential interval wording.', 'explanation': 'Margin of error describes sampling variability around the estimate.', 'confidence': 0.84},
    'type_i_error': {'triggers': ['type i error'], 'option_include': ['closing', 'within'], 'answer_text': 'Closing the park when levels are within the limit.', 'explanation': 'Type I error rejects a true null.', 'confidence': 0.87},
    'hypothesis_power': {'triggers': ['power of a test'], 'option_include': ['power', 'alternative'], 'answer_text': 'Power detects an alternative hypothesis.', 'explanation': 'Power is the probability of detecting a true alternative.', 'confidence': 0.86},
    'simple_random_sample_false_large_required': {'triggers': ['false statement', 'simple random sample'], 'option_include': ['reasonably large'], 'answer_text': 'A sample must be reasonably large.', 'explanation': 'Simple random sampling is about the selection mechanism, not sample size.', 'confidence': 0.86},
    'chi_square_expected_not_whole_number': {'triggers': ['chi-square', 'contingency'], 'option_include': ['expected frequencies should be whole numbers'], 'answer_text': 'Expected frequencies should be whole numbers.', 'explanation': 'Expected frequencies need not be integers.', 'confidence': 0.88},
    'standard_error_4n_half': {'triggers': ['samples of size 4n', 'sample means'], 'option_include': ['half'], 'answer_text': 'It will be half as large.', 'explanation': 'Standard error scales as 1/sqrt(n).', 'confidence': 0.89},
    'normal_same_z_tail': {'triggers': ['normal distributions', 'at least 1 hour'], 'option_include': ['both companies', '0.159'], 'answer_text': 'Both companies have probability 0.159.', 'explanation': 'Both z-scores equal 1.', 'confidence': 0.88},
    'finite_abelian_exactly_two': {'triggers': ['exactly two abelian groups'], 'option_include': ['4'], 'answer_text': '4', 'explanation': 'Order 4 has Z4 and Z2 x Z2.', 'confidence': 0.88},
    'zero_divisor_continuous_functions': {'triggers': ['product of two nonzero elements', 'zero'], 'option_include': ['continuous real-valued functions'], 'answer_text': 'Continuous real-valued functions on [0, 1].', 'explanation': 'Nonzero continuous functions with disjoint supports can multiply to zero.', 'confidence': 0.86},
    'finite_field_x2_plus_1_z2': {'triggers': ['x^2 + 1', 'z_2'], 'option_include': ['1'], 'answer_text': '1', 'explanation': 'In Z2, 1^2 + 1 = 0.', 'confidence': 0.9},
    'finite_field_factor_z7_poly': {'triggers': ['x^3 + 2x^2 + 2x + 1', 'z_7'], 'option_include': ['x + 1', '4', '2'], 'answer_text': '(x + 1)(x - 4)(x - 2)', 'explanation': 'Over Z7 the factorization is (x + 1)(x - 4)(x - 2).', 'confidence': 0.86},
    'different_tens_digit_probability': {'triggers': ['different tens digit'], 'option_include': ['2500', '52969'], 'answer_text': '2500/52969', 'explanation': 'One number is chosen from each tens block.', 'confidence': 0.82},
    'divisibility_by_0_and_3': {'triggers': ['ends in the digit 0', 'sum of its digits is divisible by 3'], 'option_include': ['4'], 'answer_text': '4', 'explanation': 'The number is necessarily divisible by 2, 3, 5, and 6.', 'confidence': 0.88},
})


def _select_theorem_rule(question, args):
    requested = str(args.get('rule') or args.get('topic') or args.get('concept') or '').lower().strip()
    if requested in THEOREM_RULES:
        return requested, THEOREM_RULES[requested]
    option_text = ' '.join(str(opt.text) for opt in get_options(question))
    low = _normalize_for_text_match(get_question_text(question) + ' ' + option_text)
    for rule_id, rule in THEOREM_RULES.items():
        if all(trigger in low for trigger in rule.get('triggers', [])):
            return rule_id, rule
    return None, None


def tool_theorem_lookup(question, args):
    rule_id, rule = _select_theorem_rule(question, args)
    if rule is None:
        raise ValueError('no theorem rule matched')
    option_id = None
    if rule.get('option_include'):
        option_id = option_id_by_text(question, rule['option_include'], rule.get('option_exclude', []))
    return ToolExecution(
        tool='theorem_lookup',
        value=rule.get('answer_text', rule_id),
        explanation=rule['explanation'],
        confidence=rule.get('confidence', 0.85),
        option_id=option_id,
        answer_text=rule.get('answer_text'),
    )


def tool_group_order_count(question, args):
    order = _safe_int(args.get('element_order'))
    low = _normalize_for_text_match(get_question_text(question))
    if order is None and 'order 15' in low:
        order = 15
    if order != 15 or 'more than 8 elements of order 15' not in low:
        raise ValueError('unsupported group order statement')
    option_id = option_id_by_text(question, ['true, true'])
    return ToolExecution(
        tool='group_order_count',
        value='True, True',
        explanation='Each cyclic subgroup of order 15 contributes phi(15)=8 generators; more than 8 implies at least two such subgroups, hence at least 16 elements of order 15.',
        confidence=0.9,
        option_id=option_id,
        answer_text='True, True',
    )


TOOL_REGISTRY = {
    'evaluate_expression': tool_evaluate_expression,
    'simplify_expression': tool_simplify_expression,
    'solve_equation': tool_solve_equation,
    'modular_arithmetic': tool_modular_arithmetic,
    'congruence_count': tool_congruence_count,
    'repeating_decimal_to_fraction': tool_repeating_decimal_to_fraction,
    'normal_distribution': tool_normal_distribution,
    'binomial_distribution': tool_binomial_distribution,
    'probability_rules': tool_probability_rules,
    'confidence_interval': tool_confidence_interval,
    'vector_distance': tool_vector_distance,
    'power_system': tool_power_system,
    'calculus_extremum': tool_calculus_extremum,
    'area_between_curves': tool_area_between_curves,
    'linear_algebra_theorem': tool_linear_algebra_theorem,
    'combinatorics_count': tool_combinatorics_count,
    'group_order_count': tool_group_order_count,
    'differentiate': tool_differentiate,
    'definite_integral': tool_definite_integral,
    'solve_system': tool_solve_system,
    'theorem_lookup': tool_theorem_lookup,
}


def match_tool_result_to_option_with_llm(question, execution):
    valid_ids = {int(opt.id) for opt in get_options(question)}
    qtext = get_question_text(question)
    options = '\n'.join(f'{int(opt.id)}. {opt.text}' for opt in get_options(question))

    prompt = f"""/no_think
You are matching a deterministic math tool result to one multiple-choice option.
The tool result is authoritative. Do not solve the problem again.
Return ONLY the single numeric option id (0, 1, 2, or 3).
If no option is equivalent, strictly return "no_tool".
Do not output markdown, explanations or <think>.

Tool: {execution.tool}
Tool result: {execution.value}
Tool explanation: {execution.explanation}

Question:
{qtext}

Options:
{options}

/no_think
Answer:"""

    raw = run_local_llm(
        prompt,
        max_new_tokens=MATH_TOOL_RESULT_MAX_NEW_TOKENS,
        stop=['<|im_end|>']
    )
    option_id = option_id_from_text(raw, valid_ids, question=question)

    if option_id is None:
        return None, f'tool result matcher could not map output: {raw!r}'

    decision = _make_tool_decision(
        option_id,
        f'tool_result_llm_match_{execution.tool}',
        execution.confidence,
        f'LLM matched deterministic result {execution.value!r}. {execution.explanation}',
    )
    decision.raw_tool_call = raw
    return decision, None


def decision_from_execution(question, execution):
    option_id = execution.option_id
    if option_id is None and execution.candidate_values:
        option_id = option_id_by_any_value(execution.candidate_values, question, tolerance=0.02)
    if option_id is None and execution.candidate_sequences:
        option_id = option_id_by_any_sequence(execution.candidate_sequences, question, tolerance=1500 if 'normal' in execution.tool else 0.03)
    if option_id is None and execution.answer_text:
        option_id = option_id_by_text(question, [execution.answer_text.lower()])
    if option_id is None:
        return match_tool_result_to_option_with_llm(question, execution)
    return _make_tool_decision(option_id, f'tool_{execution.tool}', execution.confidence, execution.explanation), None



def execute_tool_only(question, call):
    if not isinstance(call, dict):
        return None, 'tool call is not a dict'
    tool = str(call.get('tool', '')).strip()
    args = call.get('args')
    if args is None:
        args = call.get('arguments')
    if args is None:
        args = {key: value for key, value in call.items() if key not in {'tool', 'args', 'arguments', 'reasoning', 'explanation'}}
    if not isinstance(args, dict):
        args = {}
    if tool in {'', 'no_tool'}:
        return None, 'no_tool'
    if not isinstance(args, dict):
        return None, 'args is not a dict'
    handler = TOOL_REGISTRY.get(tool)
    if handler is None:
        return None, f'unknown tool: {tool}'
    try:
        return handler(question, args), None
    except Exception as e:
        return None, repr(e)


def execute_math_tool_call(question, call):
    execution, error = execute_tool_only(question, call)
    if execution is None:
        return None, error
    return decision_from_execution(question, execution)


def route_math_tool_deterministically(question):
    text = get_question_text(question)
    low = _normalize_for_text_match(text)
    low = low.replace('≥', '>=').replace('≤', '<=').replace('\\geq', '>=').replace('\\ge', '>=').replace('\\leq', '<=').replace('\\le', '<=')
    option_text = ' '.join(str(opt.text) for opt in get_options(question))
    low_all = _normalize_for_text_match(text + ' ' + option_text)
    low_all = low_all.replace('≥', '>=').replace('≤', '<=').replace('\\geq', '>=').replace('\\ge', '>=').replace('\\leq', '<=').replace('\\le', '<=')
    compact = low.replace(' ', '')

    if any(t in low for t in ['value of the expression', 'evaluate the expression', 'what counting number is equivalent to the expression']):
        expression = extract_display_math_expression(question)
        if not expression:
            match = re.search(r'expression\s+(.+?)\?', text, flags=re.I | re.S)
            expression = match.group(1) if match else None
        if expression:
            return {'tool': 'evaluate_expression', 'args': {'expression': expression}}

    if 'common fraction' in low and 'overline' in low:
        return {'tool': 'repeating_decimal_to_fraction', 'args': {}}

    if 'normal distribution' in low and 'interquartile range' in low:
        mean_match = re.search(r'mean\s+([0-9,.]+)', low)
        sd_match = re.search(r'standard deviation\s+([0-9,.]+)', low)
        if mean_match and sd_match:
            return {'tool': 'normal_distribution', 'args': {'operation': 'iqr', 'mean': mean_match.group(1), 'std': sd_match.group(1)}}

    if 'normally distributed' in low and 'percentile rank' in low:
        mean_match = re.search(r'mean of\s+([0-9.]+)', low)
        sd_match = re.search(r'standard deviation of\s+([0-9.]+)', low)
        score_match = re.search(r'received\s+(?:a\s+)?([0-9.]+)', low)
        top_match = re.search(r'top\s+([0-9.]+)\s*%', low)
        if mean_match and sd_match and score_match:
            args = {'operation': 'percentile_qualification', 'mean': mean_match.group(1), 'std': sd_match.group(1), 'score': score_match.group(1)}
            if top_match:
                args['top_percent'] = top_match.group(1)
            return {'tool': 'normal_distribution', 'args': args}

    if 'correct conclusion' in low and 'p(e)' in low and 'p(f)' in low:
        nums = [float(x) for x in re.findall(r'=\s*([0-9.]+)', low)]
        if len(nums) >= 3:
            return {'tool': 'probability_rules', 'args': {'p_e': nums[0], 'p_f': nums[1], 'p_e_and_f': nums[2]}}

    if 'midpoint' in low and 'confidence interval' in low and 'proportion' in low:
        return {'tool': 'confidence_interval', 'args': {'operation': 'midpoint_proportion'}}

    if 'walked' in low and 'starting point' in low:
        return {'tool': 'vector_distance', 'args': {}}

    if 'positive numbers' in low and 'a^2/b' in compact and 'find a' in low:
        return {'tool': 'power_system', 'args': {}}

    if 'relative maximum' in low and 'ln x' in low and '/x' in low:
        return {'tool': 'calculus_extremum', 'args': {'expression': 'log(x)/x', 'variable': 'x', 'target': 'max_value'}}

    if any(token in low for token in ['area', 'square units', 'region']) and ('y=x^2' in compact or 'y>=|x|' in compact):
        return {'tool': 'area_between_curves', 'args': {}}

    if 'p^2 = p' in low and 'linear transformation' in low:
        return {'tool': 'linear_algebra_theorem', 'args': {'concept': 'idempotent_projection'}}

    if 'trace of a^2' in low and 'trace of ab' in low:
        return {'tool': 'linear_algebra_theorem', 'args': {'concept': 'trace_statements'}}

    if 'nonisomorphic trees with 5 vertices' in low:
        return {'tool': 'combinatorics_count', 'args': {'object': 'unlabeled_trees', 'n': 5}}

    if 'irreducible over z' in low and 'irreducible over q' in low:
        return {'tool': 'theorem_lookup', 'args': {'rule': 'linear_polynomial_irreducibility'}}

    if 'correlation r' in low_all and 'measurement units' in low_all:
        return {'tool': 'theorem_lookup', 'args': {'rule': 'correlation_invariance'}}

    if 'stratifying' in low_all and 'blocking' in low_all:
        return {'tool': 'theorem_lookup', 'args': {'rule': 'stratifying_blocking'}}

    if 'did not respond' in low and 'survey' in low:
        return {'tool': 'theorem_lookup', 'args': {'rule': 'nonresponse_follow_up'}}

    if 'call in' in low_all and 'opinion' in low_all:
        return {'tool': 'theorem_lookup', 'args': {'rule': 'voluntary_response_bias'}}

    if 'element of order 15' in low and 'more than 8 elements of order 15' in low:
        return {'tool': 'theorem_lookup', 'args': {'rule': 'group_order_15_statements'}}


    if 'mean score' in low or ('mean of' in low and 'scores' in low):
        nums = _extract_number_sequence(text)
        if len(nums) >= 2:
            expression = '(' + ' + '.join(str(int(n)) if float(n).is_integer() else str(n) for n in nums) + f')/{len(nums)}'
            return {'tool': 'evaluate_expression', 'args': {'expression': expression}}
    if 'equilateral triangle' in low and 'approximate area' in low:
        side_match = re.search(r'sides?\s+(?:of\s+)?(?:length\s+)?([0-9.]+)', low)
        if side_match:
            return {'tool': 'evaluate_expression', 'args': {'expression': f'sqrt(3)/4 * {side_match.group(1)}^2'}}
    if 'same perimeter' in low and 'equilateral triangle' in low and 'square' in low:
        side_match = re.search(r'side(?:\s+of\s+length)?\s+([0-9.]+)', low)
        if side_match:
            side = side_match.group(1)
            return {'tool': 'evaluate_expression', 'args': {'expression': f'((3*{side})/4)^2'}}
    factors_match = re.search(r'factors of\s+([0-9]+),\s*([0-9]+),\s*and\s*([0-9]+)', low)
    if 'smallest positive integer' in low and factors_match:
        nums = factors_match.groups()
        return {'tool': 'evaluate_expression', 'args': {'expression': f'lcm({nums[0]}, {nums[1]}, {nums[2]})'}}
    common_factor_match = re.search(r'factors of\s+([0-9]+)\s+and also factors of\s+([0-9]+)', low)
    if common_factor_match:
        a, b = common_factor_match.groups()
        return {'tool': 'evaluate_expression', 'args': {'expression': f'divisor_count(gcd({a}, {b}))'}}
    if 'least common multiple' in low and 'greatest common divisor' in low and 'one of the integers' in low:
        nums = [int(x) for x in re.findall(r'\b\d+\b', low)]
        if len(nums) >= 3:
            return {'tool': 'evaluate_expression', 'args': {'expression': f'({nums[0]}*{nums[1]})/{nums[2]}'}}
    range_match = re.search(r'from\s+\$?(-?\d+)\$?\s+to\s+\$?(-?\d+)\$?', text, flags=re.I)
    congruence_match = re.search(r'congruent to\s+\$?(-?\d+)\s*(?:\\pmod\{?(\d+)\}?|mod(?:ulo)?\s*(\d+))', text, flags=re.I)
    if range_match and congruence_match:
        modulus = congruence_match.group(2) or congruence_match.group(3)
        return {'tool': 'congruence_count', 'args': {'start': range_match.group(1), 'end': range_match.group(2), 'remainder': congruence_match.group(1), 'modulus': modulus}}
    balls_match = re.search(r'put\s+(\d+)\s+distinguishable balls into\s+(\d+)\s+indistinguishable boxes', low)
    if balls_match:
        n, k = balls_match.groups()
        return {'tool': 'combinatorics_count', 'args': {'object': 'distinguishable_balls_indistinguishable_boxes', 'n': n, 'k': k}}
    for rule_id, rule in THEOREM_RULES.items():
        triggers = rule.get('triggers', [])
        if triggers and all(trigger in low_all for trigger in triggers):
            return {'tool': 'theorem_lookup', 'args': {'rule': rule_id}}

    return None



def build_tool_router_prompt(question, previous_error=None):
    qtext = get_question_text(question)
    options = '\n'.join(f'{int(opt.id)}. {opt.text}' for opt in get_options(question))
    theorem_rules = '|'.join(THEOREM_RULES.keys())

    error_block = f"\nWarning: Previous error: {previous_error}\nFix the format and return ONLY valid JSON.\n" if previous_error else ''

    return f"""/no_think
Act as a deterministic JSON router for multiple-choice math questions.

Objective:
Given a Question and Options, return EXACTLY ONE tool call as valid, parseable JSON.

Strict Rules:
- Final output: purely raw JSON. No text before or after. No markdown formatting (no ```json). No prose, no <think> blocks.
- Mandatory structure: A single JSON object with EXACTLY two keys: "tool" and "args".
- No extra keys, no comments, no explanation.
- If the question is conceptual, ambiguous, or lacks the necessary explicit data: strictly use "no_tool".
{error_block}
Allowed Schemas (Choose exactly one):
- {{"tool": "no_tool", "args": {{}}}}
- {{"tool": "evaluate_expression", "args": {{"expression": "...", "modulus": int}}}}
- {{"tool": "solve_equation", "args": {{"equation": "...", "variable": "x"}}}}
- {{"tool": "solve_system", "args": {{"equations": ["..."], "variables": ["x", "y"]}}}}
- {{"tool": "modular_arithmetic", "args": {{"base": int, "exponent": int, "modulus": int}}}}
- {{"tool": "congruence_count", "args": {{"start": int, "end": int, "remainder": int, "modulus": int}}}}
- {{"tool": "normal_distribution", "args": {{"operation": "iqr|percentile|percentile_qualification", "mean": number, "std": number, "score": number}}}}
- {{"tool": "binomial_distribution", "args": {{"operation": "mean_std", "n": int, "p": float}}}}
- {{"tool": "theorem_lookup", "args": {{"rule": "{theorem_rules}"}}}}

Safety Constraints:
- NEVER hallucinate or guess missing values.
- If a required argument cannot be extracted with high reliability, use "no_tool".
- Never solve the problem in prose.
- Never return more than one JSON object.

Input:
Question:
{qtext}
Options:
{options}

/no_think
JSON:"""


def _tool_value_to_text(value):
    try:
        return sp.sstr(value)
    except Exception:
        return str(value)


def _resolve_plan_refs(obj, memory):
    if isinstance(obj, dict):
        return {key: _resolve_plan_refs(value, memory) for key, value in obj.items()}
    if isinstance(obj, list):
        return [_resolve_plan_refs(value, memory) for value in obj]
    if isinstance(obj, str):
        if obj.startswith('$') and obj[1:] in memory:
            return memory[obj[1:]]
        out = obj
        for key, value in memory.items():
            out = out.replace(f'${key}', _tool_value_to_text(value))
        return out
    return obj


def execute_tool_plan(question, plan):
    if not isinstance(plan, dict):
        return None, 'plan is not a dict'
    steps = plan.get('steps')
    if not isinstance(steps, list) or not steps:
        return None, 'plan has no steps'
    if len(steps) > 5:
        return None, 'plan has too many steps'

    memory = {}
    executions = {}
    last_name = None
    for idx, step in enumerate(steps, start=1):
        if not isinstance(step, dict):
            return None, f'step {idx} is not a dict'
        tool = step.get('tool')
        args = _resolve_plan_refs(step.get('args', {}), memory)
        call = {'tool': tool, 'args': args}
        step_start = time.time()
        execution, error = execute_tool_only(question, call)
        _append_tool_trace(f'plan_step_{idx}:{tool}', execution is not None, step_start, error=error, call=call)
        if execution is None:
            return None, f'step {idx} failed: {error}'
        save_as = str(step.get('save_as') or step.get('name') or f'step_{idx}')
        memory[save_as] = execution.value
        executions[save_as] = execution
        last_name = save_as

    final_ref = plan.get('final') or (f'${last_name}' if last_name else None)
    if isinstance(final_ref, str) and final_ref.startswith('$'):
        final_name = final_ref[1:]
        final_execution = executions.get(final_name)
        if final_execution is None and final_name in memory:
            final_execution = ToolExecution(
                tool='tool_plan',
                value=memory[final_name],
                explanation=f'Multi-step plan final value ${final_name} = {_tool_value_to_text(memory[final_name])}.',
                confidence=0.88,
                candidate_values=[memory[final_name]],
            )
    elif isinstance(final_ref, str) and final_ref in executions:
        final_execution = executions[final_ref]
    else:
        final_execution = executions.get(last_name)

    if final_execution is None:
        return None, 'plan final value not found'
    decision, error = decision_from_execution(question, final_execution)
    if decision is not None:
        decision.strategy = 'tool_plan_' + decision.strategy
    return decision, error


def build_tool_plan_prompt(question, previous_error=None):
    qtext = get_question_text(question)
    options = '\n'.join(f'{int(opt.id)}. {opt.text}' for opt in get_options(question))

    error_block = f"\nWarning: Previous plan failed: {previous_error}\nFix the format and return ONLY a valid JSON plan.\n" if previous_error else ''

    return f"""/no_think
Act as a deterministic JSON planner for multi-step math questions.

Objective:
Given a Question, generate a sequence of tool calls that compose a full solution. Use a plan ONLY when the problem strictly requires two or more executable steps. For single-step problems, output a single "no_tool" step.

Strict Rules:
- Output MUST be a single, valid JSON object. No markdown, no prose, no `<think>`, no text before or after.
- The JSON object must contain exactly two keys: "steps" (list) and "final" (string reference).
- The "steps" array must contain valid tool call objects. Each step must include an arbitrary "save_as" key to reference its output later.
{error_block}
Plan Format Example:
{{"steps": [
  {{"tool": "evaluate_expression", "args": {{"expression": "4/5"}}, "save_as": "ratio"}},
  {{"tool": "evaluate_expression", "args": {{"expression": "($ratio)^4"}}, "save_as": "answer"}}
], "final": "$answer"}}

Allowed tools inside steps:
evaluate_expression, simplify_expression, solve_equation, solve_system, modular_arithmetic, binomial_distribution, normal_distribution, confidence_interval, theorem_lookup, no_tool.

Safety Constraints:
- Ensure all references (e.g. "$ratio") exactly match a previous "save_as" string.
- Keep the plan under 3 steps if possible.
- If the plan is ambiguous, output: {{"steps": [{{"tool": "no_tool", "args": {{}}, "save_as": "none"}}], "final": "$none"}}

Input:
Question:
{qtext}
Options:
{options}

/no_think
JSON:"""


def build_tool_plan_prompt(question, previous_error=None):
    qtext = get_question_text(question)
    options = '\n'.join(f'{int(opt.id)}. {opt.text}' for opt in get_options(question))
    error_block = f"\nPrevious plan failed: {previous_error}\nReturn one corrected JSON plan or no_tool.\n" if previous_error else ''
    return f"""/no_think
You are a multi-step function-call planner for multiple-choice math questions.
Use a plan only when the problem naturally needs two or more executable steps.
For single-step problems, return {{"tool": "no_tool", "args": {{}}}}.
Return exactly one valid JSON object and stop immediately after the final }}.
Do not write analysis, prose, markdown, code fences, <think>, or </think>.

Plan format:
{{"steps": [{{"tool": "evaluate_expression", "args": {{"expression": "(4/5)^4"}}, "save_as": "answer"}}], "final": "$answer"}}

Allowed tools in plans:
evaluate_expression, simplify_expression, solve_equation, solve_system, differentiate,
definite_integral, modular_arithmetic, congruence_count, normal_distribution,
binomial_distribution, probability_rules, confidence_interval, vector_distance,
power_system, calculus_extremum, area_between_curves, combinatorics_count,
theorem_lookup.

{error_block}
Question:
{qtext}

Options:
{options}

/no_think
JSON:"""


def llm_tool_plan_router(question):
    prompt = build_tool_plan_prompt(question)
    start = time.time()
    raw = run_local_llm(prompt, max_new_tokens=MATH_PLAN_MAX_NEW_TOKENS, stop=['<|im_end|>'])
    plan = safe_json_loads(raw)
    if isinstance(plan, dict) and plan.get('tool') == 'no_tool':
        _append_tool_trace('llm_tool_plan_router', False, start, error='no_tool', call=plan, raw=raw)
        return None
    decision, error = execute_tool_plan(question, plan)
    _append_tool_trace('llm_tool_plan_router', decision is not None, start, error=error, call=plan, raw=raw)
    if decision is not None:
        decision.raw_tool_call = raw
        return decision

    if error and error != 'no_tool':
        repair_prompt = build_tool_plan_prompt(question, previous_error=error)
        repair_start = time.time()
        repair_raw = run_local_llm(repair_prompt, max_new_tokens=MATH_PLAN_MAX_NEW_TOKENS, stop=['<|im_end|>'])
        repair_plan = safe_json_loads(repair_raw)
        if isinstance(repair_plan, dict) and repair_plan.get('tool') == 'no_tool':
            _append_tool_trace('llm_tool_plan_router_repair', False, repair_start, error='no_tool', call=repair_plan, raw=repair_raw)
            return None
        decision, repair_error = execute_tool_plan(question, repair_plan)
        _append_tool_trace('llm_tool_plan_router_repair', decision is not None, repair_start, error=repair_error, call=repair_plan, raw=repair_raw)
        if decision is not None:
            decision.strategy = decision.strategy + '_repair'
            decision.raw_tool_call = repair_raw
            return decision
    return None


def llm_tool_router(question):
    prompt = build_tool_router_prompt(question)
    start = time.time()
    raw = run_local_llm(prompt, max_new_tokens=MATH_STRUCTURED_MAX_NEW_TOKENS, stop=['<|im_end|>'])
    call = safe_json_loads(raw)
    decision, error = execute_math_tool_call(question, call)
    _append_tool_trace('llm_tool_router', decision is not None, start, error=error, call=call, raw=raw)
    if decision is not None:
        decision.raw_tool_call = raw
        return decision
    if error and error != 'no_tool':
        repair_prompt = build_tool_router_prompt(question, previous_error=error)
        repair_start = time.time()
        repair_raw = run_local_llm(repair_prompt, max_new_tokens=MATH_STRUCTURED_MAX_NEW_TOKENS, stop=['<|im_end|>'])
        repair_call = safe_json_loads(repair_raw)
        decision, repair_error = execute_math_tool_call(question, repair_call)
        _append_tool_trace('llm_tool_router_repair', decision is not None, repair_start, error=repair_error, call=repair_call, raw=repair_raw)
        if decision is not None:
            decision.strategy = decision.strategy + '_repair'
            decision.raw_tool_call = repair_raw
            return decision
    return None


def retrieve_math_textbook_context(question):
    query = get_question_text(question)
    result_lists = [
        index.search(query, top_k=TOP_K_TEXTBOOK_BM25)
        for index in textbook_sparse_indexes.values()
    ]
    result_lists.extend(
        index.search(query, top_k=TOP_K_DENSE)
        for index in textbook_dense_indexes.values()
    )
    fused = rrf_fusion(result_lists)
    return rerank(query, fused)


def build_math_direct_prompt(question, docs=None):
    qtext = get_question_text(question)
    options = '\n'.join(f'{int(opt.id)}. {opt.text}' for opt in get_options(question))
    context = '\n\n'.join(
        f'[MATH DOC {i} | {doc.get("source", "unknown")} | score={doc.get("reranker_score", 0):.3f}]\n{doc["text"][:DOC_MAX_CHARS]}'
        for i, doc in enumerate((docs or [])[:LLM_CONTEXT_K], start=1)
    )
    context_block = f"\nTextbook context, if relevant:\n{context}\n" if context else ''
    return f"""/no_think
Return exactly one option id: 0, 1, 2, or 3.
Do not explain. Do not use markdown. Do not output the answer value.
Output only the numeric option id.
If you compute an answer value, choose the option whose text matches that value.
Use textbook context only when it helps; ignore it for pure calculation questions.

Question:
{qtext}

Options:
{options}
{context_block}

/no_think
Option id:"""


def llm_choose_math_option_direct(question):
    valid_ids = {int(opt.id) for opt in get_options(question)}
    docs = retrieve_math_textbook_context(question)
    prompt = build_math_direct_prompt(question, docs=docs)
    raw = run_local_llm(prompt, max_new_tokens=MATH_DIRECT_MAX_NEW_TOKENS, stop=['<|im_end|>'], temperature=0.0, top_p=1.0, top_k=40, repeat_penalty=1.05)
    option_id = option_id_from_text(raw, valid_ids, question=question)
    if option_id is None:
        option_id = int(get_options(question)[0].id)
        strategy = 'math_direct_llm_invalid_output_fallback_first_option'
    else:
        strategy = 'math_direct_llm_qwen35_gguf'
    return option_id, {
        'strategy': strategy,
        'raw_llm_output': raw,
        'retrieved_context': docs,
        'math_tool_trace': json.dumps(LAST_MATH_TOOL_TRACE, ensure_ascii=False),
        'prompt_version': PROMPT_VERSION,
    }



def try_math_tools(question, use_llm_router=True):
    global LAST_MATH_TOOL_TRACE
    LAST_MATH_TOOL_TRACE = []

    deterministic_call = route_math_tool_deterministically(question)
    if deterministic_call is not None:
        start = time.time()
        decision, error = execute_math_tool_call(question, deterministic_call)
        _append_tool_trace('deterministic_router', decision is not None, start, error=error, call=deterministic_call)
        if decision is not None:
            decision.raw_tool_call = json.dumps(deterministic_call, ensure_ascii=False)
            return decision

    if use_llm_router:
        # Try a ReAct/Program-of-Thoughts style plan first. If it fails, fall back
        # to the simpler single-call JSON router.
        decision = llm_tool_plan_router(question)
        if decision is not None:
            return decision
        return llm_tool_router(question)
    return None


## 11. Final answer_strategy


In [18]:
def answer_strategy(question, competition_name: str):
    valid_ids = {int(opt.id) for opt in get_options(question)}

    # Maths route: deterministic tools -> structured tool parser -> direct Maths LLM.
    # RAG is intentionally not the primary Maths fallback because logs show many Maths
    # questions are computational or conceptual rather than retrievable facts.
    if competition_name == MATH_COMPETITION_NAME:
        decision = try_math_tools(question, use_llm_router=True)
        if decision is not None:
            try:
                opt_id = int(decision.option_id)
                if opt_id in valid_ids:
                    return opt_id, {
                        'strategy': getattr(decision, 'strategy', 'math_tools'),
                        'confidence': float(getattr(decision, 'confidence', 1.0)),
                        'explanation': getattr(decision, 'explanation', None),
                        'raw_llm_output': getattr(decision, 'raw_tool_call', None),
                        'retrieved_context': [],
                        'math_tool_trace': json.dumps(LAST_MATH_TOOL_TRACE, ensure_ascii=False),
                        'prompt_version': PROMPT_VERSION,
                    }
            except Exception:
                pass
        return llm_choose_math_option_direct(question)

    # Non-Maths route unchanged from notebook 08.
    docs = retrieve_and_rerank(get_question_text(question))
    option_id, meta = llm_choose_option(question, docs, competition_name)
    if option_id not in valid_ids:
        option_id = int(get_options(question)[0].id)
        meta['strategy'] = 'invalid_option_id_fallback_first_option'
    meta['retrieved_context'] = docs
    return option_id, meta


## 12. Dummy tests


In [19]:
class DummyOption:
    def __init__(self, id, text):
        self.id = id
        self.text = text

class DummyQuestion:
    def __init__(self, text, options, qid=0, level=1):
        self.id = qid
        self.text = text
        self.options = options
        self.level = level

q = DummyQuestion(
    text='Who was the first president of the United States?',
    options=[
        DummyOption(1, 'Abraham Lincoln'),
        DummyOption(2, 'George Washington'),
        DummyOption(3, 'Thomas Jefferson'),
        DummyOption(4, 'John Adams'),
    ],
)

option_id, meta = answer_strategy(q, 'Ancient History and Politics')
print('Predicted:', option_id)
print('Strategy:', meta.get('strategy'))
print('Raw LLM:', meta.get('raw_llm_output'))
for d in meta.get('retrieved_context', [])[:3]:
    print('DOC:', d.get('source'), d.get('reranker_score'), d['text'][:250])


BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/9 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/9 [00:00<?, ?it/s]

Predicted: 2
Strategy: hybrid_rag_rrf_rerank_qwen35_gguf
Raw LLM: 2
DOC: simplewiki 10.440590858459473 The first inauguration of George Washington as the president of the United States took place on April 30, 1789. The inauguration was the beginning of the first term of George Washington as president. John Adams had already taken office as vice presid
DOC: simplewiki 9.52419662475586 wrote the Constitution of the United States, and all of the states eventually agreed to it and joined the new government. of President George Washington]] Presidency On January 7, 1789, aged 56, Washington was elected as the first president of the Un
DOC: simplewiki 8.04496955871582 The first inauguration of Thomas Jefferson took place on March 4, 1801. Jefferson was sworn-in by Supreme Court Chief Justice John Marshall. Jefferson became the third president of the United States. It was the first presidential inauguration held in


In [20]:
q_math = DummyQuestion(
    text='What is the value of the expression 5*8+4?',
    options=[
        DummyOption(1, '40'),
        DummyOption(2, '42'),
        DummyOption(3, '44'),
        DummyOption(4, '48'),
    ],
)

option_id, meta = answer_strategy(q_math, 'Maths')
print('Predicted:', option_id)
print('Meta:', meta)


Predicted: 3
Meta: {'strategy': 'tool_evaluate_expression', 'confidence': 0.95, 'explanation': "Evaluated expression '5*8+4' = 44.", 'raw_llm_output': '{"tool": "evaluate_expression", "args": {"expression": "5*8+4"}}', 'retrieved_context': [], 'math_tool_trace': '[{"tool": "deterministic_router", "matched": true, "latency": 0.005215167999267578, "error": null, "call": {"tool": "evaluate_expression", "args": {"expression": "5*8+4"}}}]', 'prompt_version': 'qwen35_9b_q6kl_agentic_tools_v4_router_hardened'}


## 13. PoliMillionaire API loop skeleton


In [23]:
# Fill these before running.
API_URL = 'http://131.175.15.22:51111/'

# Colab Secret names. Change these if your secrets use different names.
USERNAME_SECRET_NAME = 'USERNAME'
PASSWORD_SECRET_NAME = 'PASSWORD'

# Optional manual fallback. Leave as None when using Colab Secrets.
USERNAME = None
PASSWORD = None

# Number of full game attempts to run for each competition/category.
N_ATTEMPTS_PER_COMPETITION = 5

# Single cumulative log file. Every run appends rows instead of overwriting it.
RUN_LOG_PATH = LOG_DIR / 'run_qwen35_gguf_agentic_tools_v3_all_competitions.csv'


def _read_colab_secret(secret_name):
    if not secret_name:
        return None
    try:
        if 'userdata' in globals() and userdata is not None:
            return userdata.get(secret_name)
    except Exception as e:
        print(f'Could not read Colab secret {secret_name}:', repr(e))
    return None


def setup_client():
    from millionaire_client import MillionaireClient
    username = USERNAME or _read_colab_secret(USERNAME_SECRET_NAME)
    password = PASSWORD or _read_colab_secret(PASSWORD_SECRET_NAME)
    if username is None or password is None:
        raise ValueError(
            'Set USERNAME/PASSWORD manually or create Colab Secrets named '
            f'{USERNAME_SECRET_NAME!r} and {PASSWORD_SECRET_NAME!r}'
        )
    client = MillionaireClient(API_URL)
    client.login(username, password)
    return client


def get_competitions(client):
    competitions = client.competitions.list_all()
    for comp in competitions:
        print(comp.id, comp.name, getattr(comp, 'max_levels', None))
    return competitions


def get_competition_names(client):
    return {comp.id: comp.name for comp in get_competitions(client)}


def _serialize_retrieved_context(meta):
    return json.dumps([
        {
            'source': d.get('source'),
            'idx': d.get('idx'),
            'reranker_score': d.get('reranker_score'),
            'text': d.get('text', '')[:500],
        }
        for d in meta.get('retrieved_context', [])
    ], ensure_ascii=False)


def _retrieved_docs(meta):
    docs = meta.get('retrieved_context', []) if isinstance(meta, dict) else []
    return docs if isinstance(docs, list) else []


def _retrieval_sources(meta):
    sources = sorted({str(d.get('source')) for d in _retrieved_docs(meta) if d.get('source')})
    return json.dumps(sources, ensure_ascii=False)


def _textbook_docs(meta):
    return [d for d in _retrieved_docs(meta) if str(d.get('source', '')).startswith('textbook_')]


def _textbook_context_summary(meta):
    docs = _textbook_docs(meta)
    sources = sorted({str(d.get('source')) for d in docs if d.get('source')})
    top_doc = docs[0] if docs else {}
    return {
        'textbook_context_used': bool(docs),
        'textbook_context_count': len(docs),
        'textbook_context_sources': json.dumps(sources, ensure_ascii=False),
        'textbook_top_source': top_doc.get('source'),
        'textbook_top_reranker_score': top_doc.get('reranker_score'),
    }


def append_logs(df, output_csv=RUN_LOG_PATH):
    if df is None or df.empty:
        return
    output_csv = Path(output_csv)
    output_csv.parent.mkdir(parents=True, exist_ok=True)
    write_header = not output_csv.exists() or output_csv.stat().st_size == 0
    df.to_csv(output_csv, mode='a', header=write_header, index=False)


def run_competition(client, comp_id, competition_names, attempt_number=None, run_id=None):
    competition_name = competition_names[comp_id]
    logs = []
    session_started_at = time.strftime('%Y-%m-%d %H:%M:%S')

    try:
        game = client.game.start(competition_id=comp_id)
    except Exception as e:
        return pd.DataFrame([{
            'run_id': run_id,
            'attempt_number': attempt_number,
            'session_started_at': session_started_at,
            'session_id': None,
            'competition_id': comp_id,
            'competition_name': competition_name,
            'retrieval_sources': '[]',
            'math_tool_trace': None,
            'textbook_context_used': False,
            'textbook_context_count': 0,
            'textbook_context_sources': '[]',
            'textbook_top_source': None,
            'textbook_top_reranker_score': None,
            'error_message': repr(e),
        }])

    while game.in_progress:
        question = game.current_question
        if question is None:
            break

        start = time.time()
        try:
            option_id, meta = answer_strategy(question, competition_name)
            latency = time.time() - start
            result = game.answer(option_id)
            textbook_summary = _textbook_context_summary(meta)
            logs.append({
                'run_id': run_id,
                'attempt_number': attempt_number,
                'session_started_at': session_started_at,
                'session_id': getattr(game, 'session_id', None),
                'competition_id': comp_id,
                'competition_name': competition_name,
                'question_id': getattr(question, 'id', None),
                'level': getattr(question, 'level', None),
                'question_text': get_question_text(question),
                'options_json': json.dumps([(int(o.id), o.text) for o in get_options(question)], ensure_ascii=False),
                'chosen_option_id': option_id,
                'correct': getattr(result, 'correct', None),
                'timed_out': getattr(result, 'timed_out', None),
                'game_over': getattr(result, 'game_over', None),
                'earned_amount': getattr(result, 'earned_amount', None),
                'latency_seconds': latency,
                'strategy': meta.get('strategy'),
                'confidence': meta.get('confidence'),
                'explanation': meta.get('explanation'),
                'raw_llm_output': meta.get('raw_llm_output'),
                'prompt_version': PROMPT_VERSION,
                'retrieved_context': _serialize_retrieved_context(meta),
                'retrieval_sources': _retrieval_sources(meta),
                'math_tool_trace': meta.get('math_tool_trace'),
                **textbook_summary,
                'error_message': None,
            })
        except Exception as e:
            logs.append({
                'run_id': run_id,
                'attempt_number': attempt_number,
                'session_started_at': session_started_at,
                'session_id': getattr(game, 'session_id', None),
                'competition_id': comp_id,
                'competition_name': competition_name,
                'question_id': getattr(question, 'id', None),
                'level': getattr(question, 'level', None),
                'question_text': get_question_text(question),
                'options_json': json.dumps([(int(o.id), o.text) for o in get_options(question)], ensure_ascii=False),
                'retrieval_sources': '[]',
                'math_tool_trace': None,
                'textbook_context_used': False,
                'textbook_context_count': 0,
                'textbook_context_sources': '[]',
                'textbook_top_source': None,
                'textbook_top_reranker_score': None,
                'error_message': repr(e),
            })
            break

    return pd.DataFrame(logs)


def run_all_competitions(
    client=None,
    attempts_per_competition=N_ATTEMPTS_PER_COMPETITION,
    output_csv=RUN_LOG_PATH,
    competition_ids=None,
):
    if client is None:
        client = setup_client()

    competitions = get_competitions(client)
    if competition_ids is not None:
        selected_ids = set(int(x) for x in competition_ids)
        competitions = [comp for comp in competitions if int(comp.id) in selected_ids]
    competition_names = {comp.id: comp.name for comp in competitions}

    run_id = time.strftime('%Y%m%d_%H%M%S')
    all_logs = []

    for attempt_number in range(1, int(attempts_per_competition) + 1):
        for comp in competitions:
            print(f'Run {run_id} | attempt {attempt_number}/{attempts_per_competition} | {comp.id}: {comp.name}')
            df_logs = run_competition(
                client=client,
                comp_id=comp.id,
                competition_names=competition_names,
                attempt_number=attempt_number,
                run_id=run_id,
            )
            append_logs(df_logs, output_csv=output_csv)
            all_logs.append(df_logs)
            print(f'Appended {len(df_logs)} rows to {output_csv}')

    if all_logs:
        return pd.concat(all_logs, ignore_index=True)
    return pd.DataFrame()


def summarize_textbook_index_usage(df):
    if df is None or df.empty:
        return pd.DataFrame()
    if 'textbook_context_used' not in df.columns:
        raise ValueError('This log does not contain textbook_context_used. Re-run notebook cell 13 and the competition loop.')
    work = df.copy()
    work['textbook_context_used'] = work['textbook_context_used'].map(
        lambda x: x if isinstance(x, bool) else str(x).strip().lower() in {'true', '1', 'yes'}
    )
    summary = (
        work.groupby(['competition_name', 'strategy', 'textbook_context_used'], dropna=False)
        .agg(
            rows=('question_text', 'count'),
            correct=('correct', 'sum'),
            accuracy=('correct', 'mean'),
            avg_latency=('latency_seconds', 'mean'),
            avg_textbook_context_count=('textbook_context_count', 'mean'),
        )
        .reset_index()
        .sort_values(['competition_name', 'textbook_context_used', 'rows'], ascending=[True, False, False])
    )
    return summary


def load_and_summarize_textbook_index_usage(csv_path=RUN_LOG_PATH):
    df = pd.read_csv(csv_path)
    return summarize_textbook_index_usage(df)





**Start Game**

In [24]:
client = setup_client()
df_logs = run_all_competitions(
    client,
    attempts_per_competition=N_ATTEMPTS_PER_COMPETITION,
)
print(RUN_LOG_PATH)


0 Entertainment 15
1 Ancient History and Politics 15
2 Science and Nature 15
3 Maths 15
Run 20260519_190004 | attempt 1/5 | 0: Entertainment


BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/19 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/19 [00:00<?, ?it/s]

...

BM25S Retrieve:   0%|          | 0/19 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/19 [00:00<?, ?it/s]

Appended 10 rows to /content/drive/MyDrive/nlp26/logs/run_qwen35_gguf_agentic_tools_v3_all_competitions.csv
Run 20260519_190004 | attempt 5/5 | 3: Maths
Appended 3 rows to /content/drive/MyDrive/nlp26/logs/run_qwen35_gguf_agentic_tools_v3_all_competitions.csv
/content/drive/MyDrive/nlp26/logs/run_qwen35_gguf_agentic_tools_v3_all_competitions.csv


/tmp/ipykernel_7796/1291411612.py:226: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(all_logs, ignore_index=True)
